# IV-1 離散時間信号とシステム（ノートブック版）

教科書の IV-1 の 6 節のプログラムを、1 冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。全部で数分です）。

| 節 | 試すこと | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 共通ライブラリ `common/filters.py` を scipy・numpy と照合する | A | `filters_check.py` |
| 2 | オイラーの公式と複素正弦波 | A | `euler_demo.py` |
| 3 | エイリアシングを予想してから聞く | B | `alias_listen.py` |
| 4 | リサンプリング | C | `resample_check.py` |
| 5 | 量子化・ディザと dB | D | `quantize_demo.py`・`db_demo.py` |
| 6 | LTI システムとたたみ込み | E | `lti_conv.py` |
| 7 | z 変換・極と零点 | F | `zplane_demo.py` |
| 8 | FIR と IIR の設計 | G・H | `fir_design.py`・`iir_design.py` |
| 9 | 位相と群遅延 | I | `phase_gd.py` |
| 10 | 統計の誤用と、確認問題の答え合わせ | J | `stats_misuse.py`・`quiz_check.py` |

- 次のセルたちが、共通ライブラリ（`common/__init__.py`・`common/audio_io.py`・`common/plotting.py`・`common/filters.py`）と IV-2 の `sounds.py` を、このノートブックの隣に `%%writefile` で書き出します。中身は教科書の IV-1・IV-2 の 6 節と同じなので、ほかのファイルは要りません。
- 各節のコードは 6 節のプログラムそのものです。変えたのは、ファイルの場所（`__file__` の代わりにノートブックの隣を使う）、図をセルの下に表示すること（`matplotlib.use("Agg")` を外し、`plt.show()` を足す）、音を再生ボタンで聞くことだけです。各節の出力はいったん `cap` に受け取って次のセルで表示し、教科書の手順の「確かめること」の主な数値を `ok()` で確かめます（合わないと「★確かめ失敗」で止まります。乱数・音源・ライブラリの版の分の幅を持たせてあります）。
- 内蔵マイクで録音する `record_alias.py`（手順 B の実機）は、Colab ではマイクが使えないので入れていません。Mac のターミナルで実行してください。
- **音を聞くときの注意**：再生ボタンは `listen_file()` が作ります。保存した wav（最大 0.3 以下）をそのままの大きさで鳴らすため、`IPython.display.Audio(..., normalize=False)` を使います。**`normalize=False` を消さないでください**（消すと最大値が 1 に引き上げられ、とても大きな音になります）。再生する前にパソコンの音量を小さくしてください。高い音が多い章なので、イヤホンでは特に注意し、聞こえない音のために音量を上げないでください。
- 話し声（6 節）は librosa の例の音源 `libri1` を使います（初回はダウンロードします。できないときは合成の声に切り替わります）。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・matplotlib・soundfile・librosa が最初から入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("librosa", "librosa"), ("soundfile", "soundfile"), ("pyroomacoustics", "pyroomacoustics")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, scipy, librosa, pyroomacoustics
print("numpy", numpy.__version__, "scipy", scipy.__version__, "librosa", librosa.__version__, "pyroomacoustics", pyroomacoustics.__version__)
import os
os.makedirs("common", exist_ok=True)
os.makedirs("out", exist_ok=True)

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
%%writefile common/__init__.py
"""第IV部の共通ライブラリ（stft・audio_io・plotting。後の章で metrics などを足す）"""

In [ ]:
%%writefile common/audio_io.py
"""第IV部の共通ライブラリ：音の読み書きと再生（IV-2 で定義）

  x, fs = load(path, fs=None)      # x は (n,)（モノラル）か (n, M)。float64、だいたい −1〜1
  save(path, x, fs)                # 16 ビットの wav で保存（はみ出す値は止めて知らせる）
  play(x, fs, gain_db=-20)         # 最大値を gain_db [dBFS] にそろえてから鳴らす（最初は小さい音で）
  x = record(sec, fs=16000)        # Mac の内蔵マイクで録音する（モノラル）

再生は sounddevice があればそれを使い、無ければ macOS の afplay で一時ファイルを鳴らす。
"""
import os
import pathlib
import shutil
import subprocess
import tempfile

import numpy as np
import soundfile as sf

__all__ = ["load", "save", "play", "record", "peak_dbfs"]


def load(path, fs=None):
    """wav・flac・ogg などを読む。fs を指定すると、その標本化周波数に変換する（librosa を使う）"""
    x, fs0 = sf.read(str(path), dtype="float64", always_2d=False)
    if fs is not None and fs != fs0:
        import librosa
        x = librosa.resample(x.T if x.ndim == 2 else x, orig_sr=fs0, target_sr=fs).T
        fs0 = fs
    return x, fs0


def peak_dbfs(x):
    """最大値の大きさ [dBFS]（1.0 が 0 dBFS）"""
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def save(path, x, fs):
    """16 ビット PCM の wav で保存する。|x| が 1 を超えるときは保存しない（音割れを防ぐ）"""
    x = np.asarray(x, dtype=float)
    if np.max(np.abs(x)) > 1.0:
        raise ValueError(f"最大値が {np.max(np.abs(x)):.3f} で 1 を超えています。小さくしてから保存してください"
                         "（例：x / np.max(np.abs(x)) * 0.5）")
    path = pathlib.Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x, int(fs), subtype="PCM_16")
    return path


def play(x, fs, gain_db=-20.0, blocking=True):
    """最大値を gain_db [dBFS] にそろえて鳴らす（既定 −20 dBFS ＝ 最大値 0.1）。
    はじめは Mac の音量を小さくし、少しずつ上げる。gain_db は 0 より大きくできない。"""
    if gain_db > 0:
        raise ValueError("gain_db は 0 以下にしてください（0 dBFS より大きくすると音が割れます）")
    x = np.asarray(x, dtype=float)
    peak = np.max(np.abs(x))
    if peak == 0:
        print("無音なので鳴らしません")
        return
    y = x / peak * 10 ** (gain_db / 20)
    try:
        import sounddevice as sd
    except ImportError:
        sd = None
    if sd is not None:
        sd.play(y.astype(np.float32), int(fs))
        if blocking:
            sd.wait()
        return
    if shutil.which("afplay") is None:
        raise RuntimeError("sounddevice も afplay も見つかりません。save() で保存して、ほかのアプリで聞いてください")
    fd, tmp = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    try:
        sf.write(tmp, y, int(fs), subtype="PCM_16")
        subprocess.run(["afplay", tmp], check=True)
    finally:
        os.remove(tmp)


def record(sec, fs=16000):
    """内蔵マイクで sec 秒録音し、(n,) の配列で返す（sounddevice が必要）。
    はじめて使うと macOS がマイクの許可を聞いてくるので許可する（項目名は macOS の版で要確認）"""
    import sounddevice as sd
    x = sd.rec(int(sec * fs), samplerate=int(fs), channels=1, dtype="float32")
    sd.wait()
    x = x[:, 0].astype(float)
    print(f"録音しました：最大 {peak_dbfs(x):.1f} dBFS")
    return x

In [ ]:
%%writefile common/plotting.py
"""第IV部の共通ライブラリ：図の設定とスペクトログラムの表示（IV-2 で定義）

  import matplotlib; matplotlib.use("Agg")      # 図をファイルに保存するだけのとき（第7章と同じ）
  from common.plotting import specshow
  specshow(X, fs, S, ax=ax, title="…")          # X は stft の結果（複素数）か、振幅・パワー

日本語のフォントは既存の章と同じ（Hiragino Sans）。
"""
import numpy as np
import matplotlib

matplotlib.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

__all__ = ["specshow", "to_db"]


def to_db(X, power=False, ref=None, floor_db=-120.0):
    """X（複素数・振幅・パワー）を dB にする。振幅は 20log10、パワーは 10log10。
    ref を与えないと、最大値を 0 dB にする"""
    A = np.abs(X)
    if power:
        D = 10 * np.log10(A + 1e-20)
    else:
        D = 20 * np.log10(A + 1e-20)
    D -= (D.max() if ref is None else (10 if power else 20) * np.log10(ref))
    return np.maximum(D, floor_db)


def specshow(X, fs, S, ax=None, power=False, db_range=80.0, fmax=None, title=None, cmap="magma",
             colorbar=True):
    """スペクトログラムを表示する。X は (F, T)。横軸はフレームの窓の中心の時刻 t·S/fs [s]、
    縦軸はビンの周波数 k·fs/L [Hz]（L = 2(F−1)）。色は最大値から db_range [dB] 下までを表示する。"""
    import matplotlib.pyplot as plt
    X = np.asarray(X)
    if X.ndim != 2:
        raise ValueError("specshow に渡す X は (F, T)。多チャネルなら X[:, :, m] のように1つ選ぶ")
    F, T = X.shape
    L = 2 * (F - 1)
    D = to_db(X, power=power)
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    df, dt = fs / L, S / fs
    # 各ビン・各フレームの値を、その中心の周りの長方形で塗る
    extent = (-dt / 2, (T - 1) * dt + dt / 2, -df / 2, (F - 1) * df + df / 2)
    im = ax.imshow(D, origin="lower", aspect="auto", extent=extent, cmap=cmap,
                   vmin=-db_range, vmax=0, interpolation="nearest")
    ax.set_xlabel("時間 [s]")
    ax.set_ylabel("周波数 [Hz]")
    if fmax is not None:
        ax.set_ylim(0, fmax)
    if title:
        ax.set_title(title)
    if colorbar:
        ax.figure.colorbar(im, ax=ax, label="dB（最大を 0）")
    return im

In [ ]:
%%writefile common/filters.py
"""第IV部の共通ライブラリ：離散時間のシステムとフィルタ（IV-1 で定義）

NumPy だけで書いた「中身の見える」実装。どれも scipy.signal の同じ名前の関数と
数値が一致することを IV-1 の各プログラムで確かめている（速さは scipy に負ける）。

  y = conv_direct(x, h)            たたみ込みの定義どおり（二重ループ）
  C = conv_matrix(h, n)            たたみ込みの行列（テプリッツ行列）。y = C @ x
  y = conv_fft(x, h)               FFT によるたたみ込み（ゼロ詰めで円状の回り込みを防ぐ）
  w, H = freqz(b, a, n, fs)        周波数応答 H(e^{jω})（scipy.signal.freqz と同じ点）
  w, gd = group_delay(b, a, n, fs) 群遅延 [標本]
  y = lfilter(b, a, x)             差分方程式（直接形 II 転置）。zi を渡すと (y, zf)
  zi = lfilter_zi(b, a)            ステップ入力の定常状態の初期値（filtfilt で使う）
  y = filtfilt(b, a, x)            前向き＋後ろ向き（ゼロ位相）。scipy の既定と同じ端の処理
  h = firwin_lowpass(numtaps, fc, fs, window="hamming")   窓関数法の低域通過 FIR
  b, a = butter(N, fc, fs, btype="lowpass", prewarp=True) 双一次変換のバターワース
  z, p, k = butter_zpk(...)        同じものを零点・極・利得で
  y = resample_poly(x, up, down)   有理数倍のリサンプリング（scipy と同じフィルタ）
  q = quantize(x, bits, dither=None, rng=None)  一様な量子化（mid-tread）とディザ

周波数の約束：fs を渡すと周波数は Hz、渡さないと正規化角周波数 ω [rad/標本]（0〜π）。
scipy の butter・firwin は fs を渡さないと「ナイキスト周波数を 1」とする別の単位になるので、
この教科書では scipy を呼ぶときも必ず fs= を渡す。
"""
import math

import numpy as np
from scipy.signal import get_window

__all__ = ["conv_direct", "conv_matrix", "conv_fft", "freqz", "group_delay", "lfilter",
           "lfilter_zi", "filtfilt", "firwin_lowpass", "butter", "butter_zpk",
           "bilinear_zpk", "resample_poly", "quantize"]


# ---------------------------------------------------------------- たたみ込み
def conv_direct(x, h):
    """y[n] = Σ_k h[k] x[n−k]（n = 0, …, len(x)+len(h)−2）。定義どおりの二重ループ"""
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    y = np.zeros(len(x) + len(h) - 1)
    for n in range(len(y)):
        for k in range(len(h)):
            if 0 <= n - k < len(x):
                y[n] += h[k] * x[n - k]
    return y


def conv_matrix(h, n):
    """長さ n の入力に h をたたみ込む (n+len(h)−1) × n の行列。C[i, j] = h[i−j]（範囲外は 0）"""
    h = np.asarray(h, dtype=float)
    C = np.zeros((n + len(h) - 1, n))
    for j in range(n):
        C[j:j + len(h), j] = h
    return C


def conv_fft(x, h):
    """FFT によるたたみ込み。長さ N ≥ len(x)+len(h)−1 までゼロ詰めしてから掛け算する
    （足りないと、後ろにはみ出た分が前に回り込む＝円状たたみ込みになる）"""
    x = np.asarray(x, dtype=float)
    h = np.asarray(h, dtype=float)
    n_out = len(x) + len(h) - 1
    N = 1 << (n_out - 1).bit_length()          # n_out 以上の2のべき乗
    return np.fft.irfft(np.fft.rfft(x, N) * np.fft.rfft(h, N), N)[:n_out]


# ---------------------------------------------------------------- 周波数応答
def _grid(n, fs):
    w = np.pi * np.arange(n) / n                # 0 ≤ ω < π（scipy.signal.freqz の既定と同じ点）
    return w, (w * fs / (2 * np.pi) if fs is not None else w)


def _poly_at(c, w):
    """Σ_m c[m] e^{−jωm} を、それぞれの ω で計算する"""
    m = np.arange(len(c))
    return np.exp(-1j * np.outer(w, m)) @ np.asarray(c, dtype=complex)


def freqz(b, a=1.0, n=512, fs=None):
    """H(e^{jω}) = B(e^{jω}) / A(e^{jω})。戻り値は (周波数, H)"""
    w, f = _grid(n, fs)
    return f, _poly_at(np.atleast_1d(b), w) / _poly_at(np.atleast_1d(a), w)


def group_delay(b, a=1.0, n=512, fs=None, eps=1e-10):
    """群遅延 τ(ω) = −dφ/dω [標本]。多項式 C(ω) = Σ c_m e^{−jωm} の群遅延は
    Re[ Σ m c_m e^{−jωm} / C(ω) ] なので、τ = τ_B − τ_A。
    B か A が 0 になる点（単位円の上の零点）では定義できないので nan を返す"""
    w, f = _grid(n, fs)
    out = np.zeros(n)
    for c, sign in ((np.atleast_1d(b), 1.0), (np.atleast_1d(a), -1.0)):
        C = _poly_at(c, w)
        dC = _poly_at(np.arange(len(c)) * np.asarray(c, dtype=float), w)
        bad = np.abs(C) < eps * np.max(np.abs(c))
        tau = np.full(n, np.nan)
        tau[~bad] = np.real(dC[~bad] / C[~bad])
        out = out + sign * tau
    return f, out


# ---------------------------------------------------------------- 差分方程式
def _norm_ba(b, a):
    b = np.atleast_1d(np.asarray(b, dtype=float))
    a = np.atleast_1d(np.asarray(a, dtype=float))
    if a[0] == 0:
        raise ValueError("a[0] が 0 です")
    b, a = b / a[0], a / a[0]
    K = max(len(a), len(b))
    return np.pad(b, (0, K - len(b))), np.pad(a, (0, K - len(a)))


def lfilter(b, a, x, zi=None):
    """a[0]y[n] + a[1]y[n−1] + … = b[0]x[n] + b[1]x[n−1] + … を、直接形 II 転置で計算する。
    z は K−1 個の内部の状態（K = max(len(a), len(b))）。zi を渡すと (y, 最後の状態) を返す"""
    b, a = _norm_ba(b, a)
    K = len(a)
    z = np.zeros(K - 1) if zi is None else np.array(zi, dtype=float)
    x = np.asarray(x, dtype=float)
    y = np.zeros(len(x))
    for n, xn in enumerate(x):
        yn = b[0] * xn + (z[0] if K > 1 else 0.0)
        for i in range(K - 2):
            z[i] = b[i + 1] * xn + z[i + 1] - a[i + 1] * yn
        if K > 1:
            z[K - 2] = b[K - 1] * xn - a[K - 1] * yn
        y[n] = yn
    return y if zi is None else (y, z)


def lfilter_zi(b, a):
    """入力が 1 のまま続いたときの、定常状態の内部の状態 zi（scipy.signal.lfilter_zi と同じ式）。
    z = A^T z + B（A は a のコンパニオン行列、B = b[1:] − a[1:]·b[0]）を z について解く"""
    b, a = _norm_ba(b, a)
    K = len(a)
    if K == 1:
        return np.zeros(0)
    comp = np.zeros((K - 1, K - 1))
    comp[0, :] = -a[1:]
    comp[1:, :-1] += np.eye(K - 2)
    return np.linalg.solve(np.eye(K - 1) - comp.T, b[1:] - a[1:] * b[0])


def filtfilt(b, a, x, padlen=None):
    """前向きにかけ、時間を逆にしてもう一度かける（ゼロ位相、振幅は |H|²）。
    端は scipy.signal.filtfilt の既定と同じ：padlen = 3·max(len(a), len(b)) 点を
    「点対称（odd）」に足し、ステップの定常状態から始める"""
    a1 = np.atleast_1d(a)
    b1 = np.atleast_1d(b)
    x = np.asarray(x, dtype=float)
    edge = 3 * max(len(a1), len(b1)) if padlen is None else padlen
    if len(x) <= edge:
        raise ValueError("信号が短すぎます（padlen より長くしてください）")
    left = 2 * x[0] - x[edge:0:-1]
    right = 2 * x[-1] - x[-2:-(edge + 2):-1]
    ext = np.concatenate([left, x, right])
    zi = lfilter_zi(b1, a1)
    y, _ = lfilter(b1, a1, ext, zi=zi * ext[0])
    y, _ = lfilter(b1, a1, y[::-1], zi=zi * y[-1])
    y = y[::-1]
    return y[edge:len(y) - edge] if edge > 0 else y


# ---------------------------------------------------------------- FIR の設計（窓関数法）
def firwin_lowpass(numtaps, fc, fs, window="hamming"):
    """理想の低域通過フィルタ h_d[n] = sin(ω_c m)/(π m)（m = n − (numtaps−1)/2）に窓を掛け、
    直流の利得が 1 になるように割る。scipy.signal.firwin(numtaps, fc, window=window, fs=fs) と同じ。
    窓は左右対称（get_window(..., fftbins=False)）。numtaps は奇数を勧める"""
    if not 0 < fc < fs / 2:
        raise ValueError("遮断周波数は 0 < fc < fs/2")
    nu = 2 * fc / fs                              # ω_c/π（0〜1）
    m = np.arange(numtaps) - (numtaps - 1) / 2
    h = nu * np.sinc(nu * m)                      # np.sinc(x) = sin(πx)/(πx)
    h = h * get_window(window, numtaps, fftbins=False)
    return h / np.sum(h)


# ---------------------------------------------------------------- IIR の設計（双一次変換）
def bilinear_zpk(z, p, k, fs):
    """s = 2fs(1 − z⁻¹)/(1 + z⁻¹) で、アナログの零点・極・利得をディジタルへ写す。
    s 平面の点 s₀ は z₀ = (2fs + s₀)/(2fs − s₀) へ。零点の足りない分は z = −1（ω = π）に置く"""
    z = np.atleast_1d(np.asarray(z, dtype=complex))
    p = np.atleast_1d(np.asarray(p, dtype=complex))
    c = 2.0 * fs
    zd = (c + z) / (c - z)
    pd = (c + p) / (c - p)
    zd = np.concatenate([zd, -np.ones(len(p) - len(z))])
    kd = k * np.real(np.prod(c - z) / np.prod(c - p))
    return zd, pd, kd


def butter_zpk(N, fc, fs, btype="lowpass", prewarp=True):
    """N 次のバターワース（低域・高域）。アナログの原型の極 e^{jπ(2k+N−1)/(2N)}（左半面）を
    遮断角周波数 Ωc に広げてから双一次変換する。prewarp=True なら Ωc = 2fs·tan(π fc/fs)
    （ディジタルで −3 dB の点がちょうど fc に来る）、False なら Ωc = 2π fc（ずれる）"""
    k_idx = np.arange(1, N + 1)
    proto = np.exp(1j * np.pi * (2 * k_idx + N - 1) / (2 * N))
    Wc = 2 * fs * np.tan(np.pi * fc / fs) if prewarp else 2 * np.pi * fc
    if btype == "lowpass":
        z, p, k = np.array([]), Wc * proto, Wc ** N
    elif btype == "highpass":
        z, p, k = np.zeros(N), Wc / proto, 1.0
    else:
        raise ValueError("btype は lowpass か highpass")
    return bilinear_zpk(z, p, np.real(k), fs)


def butter(N, fc, fs, btype="lowpass", prewarp=True):
    """butter_zpk を多項式の係数 (b, a) にする。次数が高い（目安 8 以上）・遮断周波数が低いときは
    係数の丸め誤差で不安定になりやすいので、実際の処理では scipy の output="sos" を使う（IV-1-3-11）"""
    z, p, k = butter_zpk(N, fc, fs, btype, prewarp)
    return np.real(k * np.poly(z)), np.real(np.poly(p))


# ---------------------------------------------------------------- リサンプリング
def resample_poly(x, up, down, window=("kaiser", 5.0)):
    """標本化周波数を up/down 倍にする。(1) 標本の間に up−1 個の 0 を入れ、
    (2) 低域通過 FIR（遮断 = 元と新しいもののナイキスト周波数の低いほう）をかけて up 倍し、
    (3) down 個に1個を取り出す。フィルタは scipy.signal.resample_poly の既定と同じ
    （長さ 2·10·max(up, down)+1、カイザー窓 β = 5）。遅れ（フィルタの中心）は取り除く。

    (1)〜(3) をそのまま計算すると、0 との掛け算と、捨てる出力の計算がほとんどで遅い
    （44.1 k → 16 k では up = 160 なので 99% 以上がむだ）。そこで、取り出す出力 m だけを
        y[m] = Σ_i x[i]·h[m·down + half − i·up]（h の範囲に入る i だけ）
    と計算する（ポリフェーズ実装の考え方。結果は (1)〜(3) と同じ）"""
    g = math.gcd(up, down)
    up, down = up // g, down // g
    x = np.asarray(x, dtype=float)
    if up == down == 1:
        return x.copy()
    r = max(up, down)
    half = 10 * r
    fs_up = 2.0 * r                                # 「ナイキスト周波数 = r」とした単位で設計する
    h = firwin_lowpass(2 * half + 1, 1.0, fs_up, window=window) * up
    n_out = -(-len(x) * up // down)                # 切り上げ
    pos = np.arange(n_out) * down + half           # 出力 m が、ゼロを入れた列のどこに当たるか（＋遅れ half）
    J = (2 * half) // up + 1                       # 1 つの出力に効く入力の数（最大）
    i = pos[:, None] // up - np.arange(J)[None, :]  # 効く入力の番号 i（新しい順）
    idx = pos[:, None] - i * up                    # そのときの h の番号（0〜2·half に入るものだけ使う）
    ok = (idx <= 2 * half) & (i >= 0) & (i < len(x))
    terms = np.where(ok, h[np.clip(idx, 0, 2 * half)] * x[np.clip(i, 0, len(x) - 1)], 0.0)
    return terms.sum(axis=1)


# ---------------------------------------------------------------- 量子化
def quantize(x, bits, dither=None, rng=None):
    """−1〜1 の信号を bits ビットで量子化する（mid-tread：0 が目盛りの上）。
    目盛り Δ = 2/2^bits、値は −1 〜 1−Δ に止める。
    dither="rect"：±Δ/2 の一様乱数、"tpdf"：±Δ の三角分布の乱数（一様2つの和）を、丸める前に足す"""
    x = np.asarray(x, dtype=float)
    d = 2.0 / 2 ** bits
    if dither is not None:
        rng = np.random.default_rng() if rng is None else rng
        if dither == "rect":
            x = x + rng.uniform(-d / 2, d / 2, x.shape)
        elif dither == "tpdf":
            x = x + rng.uniform(-d / 2, d / 2, x.shape) + rng.uniform(-d / 2, d / 2, x.shape)
        else:
            raise ValueError("dither は None・'rect'・'tpdf'")
    return np.clip(d * np.round(x / d), -1.0, 1.0 - d)

In [ ]:
%%writefile sounds.py
"""IV-2：演習で使う音（合成音と、librosa の例の音源・自分の録音）

  x = synth_vowel(fs, sec)   声に似た合成音（基本周波数 120→180 Hz、フォルマント 700・1200・2600 Hz）
  x = pluck(fs, sec, f0)     弦をはじいた音に似た合成音（Karplus–Strong 法）
  x, name = speech(fs, wav)  wav を指定すればそれ、無ければ librosa の例 libri1（ダウンロードが必要）、
                             それもだめなら synth_vowel
"""
import numpy as np
from scipy.signal import lfilter


def synth_vowel(fs=16000, sec=2.0, f0=(120.0, 180.0), formants=((700, 80), (1200, 90), (2600, 120))):
    """声帯の音（パルス列）を、声道の共振（2次の共振器を3つ直列）に通した「あ」に近い音"""
    n = int(sec * fs)
    f = np.linspace(f0[0], f0[1], n)                       # 基本周波数をゆっくり上げる
    phase = np.cumsum(f / fs)
    src = np.diff(np.floor(phase), prepend=0.0)            # 1周期に1回だけ 1 になるパルス列
    y = src
    for fc, bw in formants:                                # 中心 fc [Hz]・帯域幅 bw [Hz] の共振器
        r = np.exp(-np.pi * bw / fs)
        a = [1.0, -2 * r * np.cos(2 * np.pi * fc / fs), r * r]
        y = lfilter([1.0 - r], a, y)
    env = np.minimum(1.0, np.minimum(np.arange(n), n - np.arange(n)) / (0.05 * fs))   # 前後 50ms で立ち上げ・下げ
    y = y * env
    return y / np.max(np.abs(y)) * 0.5


def pluck(fs=16000, sec=2.0, f0=196.0, seed=0):
    """Karplus–Strong 法：雑音を1周期分の遅延線に入れ、平均をとりながら回す（弦の減衰に似る）"""
    rng = np.random.default_rng(seed)
    n = int(sec * fs)
    P = int(round(fs / f0))
    buf = rng.uniform(-1, 1, P)
    y = np.zeros(n)
    for i in range(n):
        y[i] = buf[i % P]
        buf[i % P] = 0.996 * 0.5 * (buf[i % P] + buf[(i + 1) % P])
    return y / np.max(np.abs(y)) * 0.5


def speech(fs=16000, wav=None, sec=None):
    """話し声を返す。(x, 名前)"""
    from common.audio_io import load
    if wav:
        x, _ = load(wav, fs)
        if x.ndim == 2:
            x = x[:, 0]
        name = "own"
    else:
        try:
            import librosa
            x, _ = librosa.load(librosa.ex("libri1"), sr=fs, mono=True)
            name = "libri1"
        except Exception as e:                              # ダウンロードできないときなど
            print(f"librosa の例の音源が使えないので合成音にします（{type(e).__name__}）")
            x, name = synth_vowel(fs, 3.0), "synth_vowel"
    if sec is not None:
        x = x[:int(sec * fs)]
    return x / (np.max(np.abs(x)) + 1e-12) * 0.5, name

In [ ]:
import sys, warnings, logging
sys.path.insert(0, ".")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告を出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
import numpy as np
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from common import plotting                      # import すると Hiragino Sans が設定される
from common import filters
matplotlib.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 本文の候補（0-1 のセルで用意したフォントが使われる）
matplotlib.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", message="Glyph")
print("filters.py を読み込めました：", ", ".join(filters.__all__))


def listen_file(path):
    """保存した wav をそのままの大きさで再生ボタンにする。normalize=False を消さない"""
    x, fs = sf.read(path)
    assert np.max(np.abs(x)) <= 0.31, "大きすぎる音は鳴らさない"
    display(Audio(x.T if x.ndim > 1 else x, rate=fs, normalize=False))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, i=1, text=None):
    """直前のセルの出力（cap.stdout）から、pattern の (…) の数を取り出す"""
    import re
    m = re.search(pattern, cap.stdout if text is None else text, flags=re.M)
    assert m, "出力に見つからない：" + pattern
    return float(m.group(i).replace("−", "-"))   # 出力の「−」（全角のマイナス）も数にする


_ok, _num = ok, num   # 各節のプログラムが同じ名前を使っても、確かめのセルで戻す

## 1. 共通ライブラリの照合（手順 A の 1、`filters_check.py`）

`common/filters.py` の関数（たたみ込み・周波数応答・群遅延・lfilter・filtfilt・窓関数法の FIR・バターワース・双一次変換・リサンプリング・量子化）を、scipy・numpy の同じ名前の関数と比べます。13 行がすべて「一致」で、最後に「すべて一致しました」と出れば合格です。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1：共通ライブラリ common/filters.py を scipy・numpy と照合する（直したら必ず再実行する）

すべての行が「一致」なら OK。許容誤差は、倍精度の丸め誤差の積み重なりを見込んだ値。
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
from common import filters as F

rng = np.random.default_rng(0)
fs = 16000
rows = []


def check(name, err, tol):
    rows.append((name, err, tol))
    print(f"   {'一致' if err <= tol else '★不一致'}  {name:44s} 最大の差 {err:.1e}（許容 {tol:.0e}）")


x = rng.standard_normal(4000)
h = rng.standard_normal(64)
check("conv_direct と numpy.convolve（x は 500 点）", np.max(np.abs(F.conv_direct(x[:500], h) - np.convolve(x[:500], h))), 1e-12)
check("conv_matrix @ x と numpy.convolve", np.max(np.abs(F.conv_matrix(h, 500) @ x[:500] - np.convolve(x[:500], h))), 1e-12)
check("conv_fft と scipy.signal.fftconvolve", np.max(np.abs(F.conv_fft(x, h) - ss.fftconvolve(x, h))), 1e-12)
b, a = ss.butter(4, 1000, fs=fs)
check("freqz と scipy.signal.freqz", np.max(np.abs(F.freqz(b, a, 512)[1] - ss.freqz(b, a, 512)[1])), 1e-12)
f1, g1 = F.group_delay(b, a, 512, fs=fs)
_, g2 = ss.group_delay((b, a), 512, fs=fs)
sel = f1 < 0.9 * fs / 2
check("group_delay と scipy.signal.group_delay（ω < 0.9π）", np.max(np.abs(g1[sel] - g2[sel])), 1e-9)
check("lfilter と scipy.signal.lfilter", np.max(np.abs(F.lfilter(b, a, x) - ss.lfilter(b, a, x))), 1e-12)
check("lfilter_zi と scipy.signal.lfilter_zi", np.max(np.abs(F.lfilter_zi(b, a) - ss.lfilter_zi(b, a))), 1e-12)
check("filtfilt と scipy.signal.filtfilt", np.max(np.abs(F.filtfilt(b, a, x) - ss.filtfilt(b, a, x))), 1e-12)
err = 0.0
for win in ("boxcar", "hann", "hamming", "blackman", ("kaiser", 8.0)):
    for N in (31, 101, 255):
        err = max(err, np.max(np.abs(F.firwin_lowpass(N, 3000, fs, win) - ss.firwin(N, 3000, window=win, fs=fs))))
check("firwin_lowpass と scipy.signal.firwin（5 窓×3 長さ）", err, 1e-14)
err_b, err_p = 0.0, 0.0
for N in range(1, 9):
    for bt in ("lowpass", "highpass"):
        b1, a1 = F.butter(N, 1000, fs, btype=bt)
        b2, a2 = ss.butter(N, 1000, btype=bt, fs=fs)
        err_b = max(err_b, np.max(np.abs(b1 - b2)) / np.max(np.abs(b2)), np.max(np.abs(a1 - a2)))
        p1 = F.butter_zpk(N, 1000, fs, btype=bt)[1]
        p2 = ss.butter(N, 1000, btype=bt, fs=fs, output="zpk")[1]
        err_p = max(err_p, max(np.min(np.abs(p2 - q)) for q in p1))
check("butter と scipy.signal.butter（1〜8 次、低域・高域）", err_b, 1e-13)
check("butter_zpk の極と scipy（zpk）", err_p, 1e-13)
err = 0.0
for up, down, n in ((1, 3, 48000), (3, 1, 16000), (2, 3, 9999), (160, 441, 44100), (441, 160, 16000)):
    xx = rng.standard_normal(n)
    err = max(err, np.max(np.abs(F.resample_poly(xx, up, down) - ss.resample_poly(xx, up, down))))
check("resample_poly と scipy.signal.resample_poly（5 組）", err, 1e-13)
q = F.quantize(np.linspace(-1.2, 1.2, 10001), 8)
D = 2 / 256
check("quantize：値が Δ の整数倍・範囲 −1〜1−Δ", max(np.max(np.abs(q / D - np.round(q / D))),
                                                  max(0, q.max() - (1 - D)), max(0, -1 - q.min())), 1e-12)
bad = [r for r in rows if r[1] > r[2]]
print("すべて一致しました" if not bad else f"★ {len(bad)} 行が許容を超えました")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
n_ok = len(re.findall(r"^\s+一致\s", cap.stdout, flags=re.M))
ok(n_ok == 13 and "不一致" not in cap.stdout and "すべて一致しました" in cap.stdout, f"[A] 13 行がすべて「一致」で、最後に「すべて一致しました」（一致 {n_ok} 行）")

## 2. オイラーの公式と複素正弦波（IV-1-3-1、手順 A、`euler_demo.py`）

$e^{j\theta} = \cos\theta + j\sin\theta$ をテイラー級数の部分和で確かめ、実数の正弦波が左回りと右回りの複素正弦波の和（負の周波数）であることを FFT で見ます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 A：オイラーの公式と複素正弦波（IV-1-3-1）

  1. テイラー級数の部分和で e^{jθ} を作り、cos θ + j sin θ と比べる
  2. 実数の正弦波 = 左回りと右回りの複素正弦波の和（負の周波数）
  3. フェーザ：同じ周波数の正弦波の足し算は、複素数の足し算
  4. 離散時間の複素正弦波：ω と ω + 2π は同じ数列、cos では ω と 2π − ω も同じ
  5. 離散時間の正弦波が周期的になる条件（ω/2π が有理数）
  out/euler.png：回る点と、その実部・虚部
"""
import math
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
from common import plotting  # noqa: F401  （日本語のフォントの設定）

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)


def exp_taylor(z, K):
    """e^z = Σ_{k=0}^{K−1} z^k / k! の部分和"""
    return sum(z ** k / math.factorial(k) for k in range(K))


print("1. テイラー級数の部分和 Σ (jθ)^k/k! と cos θ + j sin θ（θ = π）")
theta = np.pi
target = np.cos(theta) + 1j * np.sin(theta)
for K in (2, 4, 8, 12, 16, 20, 30):
    s = exp_taylor(1j * theta, K)
    print(f"   項の数 {K:2d}: {s.real:+.10f} {s.imag:+.10f}j   誤差 {abs(s - target):.2e}")
print(f"   e^(jπ) + 1 = {np.exp(1j * np.pi) + 1:.2e}（オイラーの等式。0 にならないのは丸め誤差）")

print("\n2. cos は左回りと右回りの和：cos(2π·1000·t) = (e^{+j…} + e^{−j…})/2")
fs = 16000
t = np.arange(fs) / fs
c = np.cos(2 * np.pi * 1000 * t)
both = (np.exp(1j * 2 * np.pi * 1000 * t) + np.exp(-1j * 2 * np.pi * 1000 * t)) / 2
print(f"   最大の差 {np.max(np.abs(c - both)):.2e}")
X = np.fft.fft(np.exp(-1j * 2 * np.pi * 1000 * t))          # 右回り（負の周波数）だけ
fr = np.fft.fftfreq(fs, 1 / fs)
print(f"   e^(−j2π·1000t) の FFT の山の周波数：{fr[np.argmax(np.abs(X))]:.0f} Hz（負の周波数）")
Xc = np.abs(np.fft.fft(c))
print(f"   cos の FFT の山：{sorted(fr[np.argsort(Xc)[-2:]].tolist())} Hz（+と−の2本、高さは半分ずつ）")

print("\n3. フェーザ：cos(ωn) + cos(ωn + π/2) = √2·cos(ωn + π/4)")
w = 2 * np.pi * 440 / fs
n = np.arange(400)
P = 1 * np.exp(1j * 0) + 1 * np.exp(1j * np.pi / 2)          # 複素振幅（フェーザ）の和
lhs = np.cos(w * n) + np.cos(w * n + np.pi / 2)
rhs = np.abs(P) * np.cos(w * n + np.angle(P))
print(f"   フェーザの和 = {P:.4f}、大きさ {abs(P):.4f}（√2 = {np.sqrt(2):.4f}）、角度 {np.degrees(np.angle(P)):.1f}°")
print(f"   波形の最大の差 {np.max(np.abs(lhs - rhs)):.2e}")

print("\n4. 離散時間では ω と ω + 2π は区別できない（エイリアシングの正体）")
n = np.arange(64)
for w0 in (0.3 * np.pi, 0.75 * np.pi):
    d1 = np.max(np.abs(np.exp(1j * w0 * n) - np.exp(1j * (w0 + 2 * np.pi) * n)))
    d2 = np.max(np.abs(np.cos(w0 * n) - np.cos((2 * np.pi - w0) * n)))
    d3 = np.max(np.abs(np.exp(1j * w0 * n) - np.exp(1j * (2 * np.pi - w0) * n)))
    print(f"   ω = {w0 / np.pi:.2f}π：e^(jωn) と e^(j(ω+2π)n) の差 {d1:.1e}、"
          f"cos(ωn) と cos((2π−ω)n) の差 {d2:.1e}、e^(jωn) と e^(j(2π−ω)n) の差 {d3:.2f}")
print("   → 複素正弦波なら区別できるのは 2π の幅、実数の cos では 0〜π（0〜fs/2）の幅だけ")

print("\n5. 周期的になるか：cos(ω n) の最小の周期（1000 点まで探す）")
for label, w0 in (("2π·3/16", 2 * np.pi * 3 / 16), ("2π·1000/16000", 2 * np.pi * 1000 / 16000),
                  ("2π·440/16000", 2 * np.pi * 440 / 16000), ("1 rad", 1.0)):
    x = np.cos(w0 * np.arange(3000))
    period = next((N for N in range(1, 1001) if np.max(np.abs(x[N:N + 1000] - x[:1000])) < 1e-9), None)
    print(f"   ω = {label:14s}: 周期 {period if period else '1000 点以内に無い（ω/2π が無理数なら周期的にならない）'}")

# 図：回る点とその影
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
th = np.linspace(0, 2 * np.pi, 400)
ax[0].plot(np.cos(th), np.sin(th), color="0.7")
for k in range(8):
    a = 2 * np.pi * k / 8
    ax[0].plot([0, np.cos(a)], [0, np.sin(a)], color="C0", alpha=0.3)
    ax[0].plot(np.cos(a), np.sin(a), "o", color="C0")
    ax[0].text(1.12 * np.cos(a), 1.12 * np.sin(a), f"n={k}", ha="center", va="center", fontsize=8)
ax[0].set_aspect("equal")
ax[0].set_title("e^{jωn}（ω = 2π/8）：単位円を左回りに回る点")
ax[0].set_xlabel("実部")
ax[0].set_ylabel("虚部")
nn = np.arange(17)
ax[1].stem(nn, np.cos(2 * np.pi * nn / 8), linefmt="C0-", markerfmt="C0o", basefmt="0.7", label="実部 cos")
ax[1].stem(nn, np.sin(2 * np.pi * nn / 8), linefmt="C1--", markerfmt="C1s", basefmt="0.7", label="虚部 sin")
ax[1].set_xlabel("n")
ax[1].legend()
ax[1].set_title("回る点の影：実部が cos、虚部が sin")
fig.tight_layout()
fig.savefig(OUT / "euler.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print(f"\n図を保存しました：{OUT / 'euler.png'}")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
e8, e16, e30 = (num(rf"項の数 +{k}: .*誤差 ([\d.e+-]+)") for k in (8, 16, 30))
ok(abs(e8 - 0.22) < 0.01 and 3e-6 < e16 < 6e-6 and e30 < 1e-14, f"[A] 部分和の誤差が項の数 8・16・30 で 0.22・4.2e-6・3e-16 と減る（{e8:.2e}・{e16:.2e}・{e30:.2e}）")
ok(num(r"FFT の山の周波数：([-\d.]+) Hz") == -1000, "[A] e^(−j2π·1000t) の FFT の山が −1000 Hz")
d03 = num(r"ω = 0\.30π：e\^\(jωn\) と e\^\(j\(ω\+2π\)n\) の差 ([\d.e+-]+)")
ok(d03 < 1e-12, f"[A] ω = 0.3π と 2.3π の差が 1e-14 の桁（{d03:.1e}）")
per = re.findall(r"周期 (\d+)$", cap.stdout, flags=re.M)
ok([int(v) for v in per] == [16, 16, 400] and "周期 1000 点以内に無い" in cap.stdout, f"[A] 周期が 16・16・400・「無い」（{per}）")

## 3. エイリアシングを予想してから聞く（IV-1-3-2・3-3、手順 B、`alias_listen.py`）

**実行する前に**、$f_s = 8000$ Hz で標本化した正弦波が何 Hz に聞こえるかを、ノートに予想して書いてください。チャープ（0 → 16 kHz）を 4 通りに標本化して、後半のエネルギーを比べます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 B：エイリアシングを予想してから聞く（IV-1-3-2・3-3）

  1. 正弦波を fs = 8000 Hz で標本化したときの「聞こえるはずの周波数」を式で予想し、FFT で確かめる
  2. 0 → 16 kHz のチャープ（周波数が上がり続ける音）を、
       (a) 48 kHz で作ったまま、(b) 16 kHz で直接作る（＝フィルタなしで標本化）、
       (c) 48 kHz → 3 個に1個を取り出す（フィルタなしの間引き）、
       (d) 48 kHz → resample_poly で 16 kHz（アンチエイリアスフィルタつき）
     の4通りで保存し、後半のエネルギーを比べる。
     (e) 同じチャープを 12 kHz で直接標本化したもの（手順の「予想」用。本文には答えを書いていない）
  3. のこぎり波（倍音が多い）を 16 kHz → 4 kHz に、フィルタなし・ありで変換して聞き比べる
  保存する wav は最大 0.3（約 −10.5 dBFS）以下にそろえる（save_quiet）。聞く前に Mac の音量を小さくする。
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
from scipy.signal import resample_poly
from common.audio_io import save

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
PEAK = 0.3


def alias_freq(f, fs):
    """周波数 f [Hz] の実数の正弦波を fs で標本化したときに見える周波数（0〜fs/2）"""
    r = f % fs                        # fs の整数倍は区別できない（ω と ω+2π が同じ）
    return min(r, fs - r)             # 実数の正弦波では f と fs − f も同じ（cos が偶関数）


def peak_freq(x, fs):
    """hann 窓の FFT の最大の山の周波数（放物線で補間）"""
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))
    k = int(np.argmax(X))
    if 0 < k < len(X) - 1:
        a, b, c = np.log(X[k - 1:k + 2] + 1e-300)
        k = k + 0.5 * (a - c) / (a - 2 * b + c)
    return k * fs / len(x)


def tone(f, fs, sec=1.0):
    n = np.arange(int(sec * fs))
    env = np.minimum(1.0, np.minimum(n, n[::-1]) / (0.02 * fs))      # 前後 20 ms で立ち上げ・下げ
    return PEAK * env * np.sin(2 * np.pi * f * n / fs)


def chirp(fs, f1=16000.0, sec=4.0):
    """瞬時周波数が 0 から f1 [Hz] まで直線で上がる音。位相 = 2π·(f1/(2·sec))·t²"""
    t = np.arange(int(sec * fs)) / fs
    env = np.minimum(1.0, np.minimum(t, sec - t) / 0.02)       # 時刻で決めるので、どの fs でも同じ形
    return PEAK * env * np.sin(np.pi * f1 / sec * t ** 2)


def save_quiet(path, x, fs):
    """最大値が PEAK を超えるときだけ PEAK に縮めて保存する（フィルタの行き過ぎで 0.3 を少し超えることがある）"""
    m = np.max(np.abs(x))
    save(path, x * (PEAK / m if m > PEAK else 1.0), fs)


def energy_db(x):
    return 10 * np.log10(np.sum(x ** 2) + 1e-30)


print("1. fs = 8000 Hz で標本化した正弦波：予想（式）と、FFT で測った周波数")
fs = 8000
print("   元の周波数 [Hz]   予想 [Hz]   測定 [Hz]")
for f in (1000, 3000, 3900, 5000, 6500, 7000, 9000, 11000, 12500, 15000, 17000):
    x = tone(f, fs)                    # 連続時間の sin(2πft) を t = n/fs で読む ＝ フィルタなしの標本化
    print(f"   {f:12d}   {alias_freq(f, fs):9.0f}   {peak_freq(x, fs):9.1f}")
    save(OUT / f"tone_{f:05d}_fs8000.wav", x, fs)
print("   → 7000・9000・15000・17000 Hz は、どれも 1000 Hz と同じ数列になる（聞いて確かめる）")

print("\n2. 0 → 16 kHz のチャープ（4 秒）")
xa = chirp(48000)
xb = chirp(16000)
xc = xa[::3]
xd = resample_poly(xa, 1, 3)
save(OUT / "chirp_a_48k.wav", xa, 48000)
save(OUT / "chirp_b_direct16k.wav", xb, 16000)
save(OUT / "chirp_c_naive16k.wav", xc, 16000)
save_quiet(OUT / "chirp_d_aa16k.wav", xd, 16000)
xe = chirp(12000)
save(OUT / "chirp_e_direct12k.wav", xe, 12000)
print(f"   (b) と (c) の最大の差 {np.max(np.abs(xb - xc)):.2e}（フィルタなしの間引き ＝ 16 kHz で直接標本化）")
i0, i1, i2 = int(1.0 * 16000), int(1.75 * 16000), int(2.25 * 16000)   # 元の周波数 4k・7k・9k の時刻
for name, x in (("(b) 直接 16 kHz", xb), ("(c) 3個に1個", xc), ("(d) resample_poly", xd)):
    e_pass = energy_db(x[i0:i1]) - 10 * np.log10(i1 - i0)          # 1 点あたりのエネルギー
    e_stop = energy_db(x[i2:]) - 10 * np.log10(len(x) - i2)
    print(f"   {name:18s}: 元が 9〜16 kHz の区間 − 元が 4〜7 kHz の区間 = {e_stop - e_pass:7.1f} dB")
t_check = (1.0, 2.5, 3.5)
for tt in t_check:
    i = int(tt * 16000)
    seg = xb[i - 512:i + 512]
    f_true = 16000 / 4.0 * tt
    print(f"   t = {tt:.1f} s：元の周波数 {f_true:6.0f} Hz → 予想 {alias_freq(f_true, 16000):6.0f} Hz、"
          f"(b) の測定 {peak_freq(seg, 16000):7.1f} Hz")
print("   (e) 12 kHz で直接標本化したチャープ：")
for tt in (1.0, 1.5, 2.5, 3.5):
    i = int(tt * 12000)
    seg = xe[i - 384:i + 384]
    f_true = 16000 / 4.0 * tt
    print(f"   t = {tt:.1f} s：元の周波数 {f_true:6.0f} Hz → 予想 {alias_freq(f_true, 12000):6.0f} Hz、"
          f"(e) の測定 {peak_freq(seg, 12000):7.1f} Hz")

print("\n3. のこぎり波（基本 440 Hz、倍音は 8 kHz 未満まで）を 16 kHz → 4 kHz")
fs = 16000
n = np.arange(2 * fs)
saw = np.zeros(len(n))
K = int(7999 // 440)
for k in range(1, K + 1):                       # 帯域制限したのこぎり波：Σ sin(2πk f0 t)/k
    saw += np.sin(2 * np.pi * k * 440 * n / fs) / k
saw = PEAK * saw / np.max(np.abs(saw))
naive = saw[::4]
aa = resample_poly(saw, 1, 4)
save(OUT / "saw_16k.wav", saw, fs)
save(OUT / "saw_naive4k.wav", naive, 4000)
save_quiet(OUT / "saw_aa4k.wav", aa, 4000)
print(f"   倍音は {K} 本（440〜{K * 440} Hz）。4 kHz で残せるのは 2000 Hz 未満の {int(1999 // 440)} 本")
print("   フィルタなしで 4 kHz にしたとき、2 kHz を超える倍音の行き先（予想）：")
print("   " + "、".join(f"{k * 440}→{alias_freq(k * 440, 4000):.0f}" for k in range(5, K + 1)))
X = np.abs(np.fft.rfft(naive * np.hanning(len(naive))))
fr = np.fft.rfftfreq(len(naive), 1 / 4000)
big = fr[(X > 0.05 * X.max())]
inharm = sorted({round(f) for f in big if min(f % 440, 440 - f % 440) > 15})
print(f"   測定：440 Hz の整数倍でない山（5% 以上）{inharm[:12]} …")
print(f"   resample_poly 版で 440 の整数倍でない成分の割合：", end="")
Xa = np.abs(np.fft.rfft(aa * np.hanning(len(aa)))) ** 2
mask = np.array([min(f % 440, 440 - f % 440) > 15 for f in fr])
Xn = X ** 2
print(f"{10 * np.log10(Xa[mask].sum() / Xa.sum()):.1f} dB（フィルタなし {10 * np.log10(Xn[mask].sum() / Xn.sum()):.1f} dB）")
print(f"\nwav を {OUT} に保存しました。音量を小さくしてから afplay で聞く")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
rows = [(int(a), int(b), float(c)) for a, b, c in re.findall(r"^\s+(\d+)\s+(\d+)\s+([\d.]+)$", cap.stdout, flags=re.M)]
pred = {a: b for a, b, c in rows}
ok(len(rows) == 11 and all(abs(b - c) <= 0.1 for a, b, c in rows), f"[B] 1. の表の「予想」と「測定」が 0.1 Hz 以内で一致（{len(rows)} 行）")
ok(pred[6500] == 1500 and pred[11000] == 3000 and pred[12500] == 3500, "[B] 6500 → 1500 Hz、11000 → 3000 Hz、12500 → 3500 Hz")
ok(num(r"\(b\) と \(c\) の最大の差 ([\d.e+-]+)") < 1e-12, "[B] 2. で (b) と (c) の差が 0")
db_b, db_d = num(r"\(b\) 直接 16 kHz\s*:.*=\s+([-\d.]+) dB"), num(r"\(d\) resample_poly\s*:.*=\s+([-\d.]+) dB")
ok(abs(db_b) < 1 and abs(db_d + 50) < 3, f"[B] (d) だけ「元が 9〜16 kHz の区間」が約 −50 dB（(b) {db_b} dB、(d) {db_d} dB）")
fe = [float(v) for v in re.findall(r"\(e\) の測定\s+([\d.]+) Hz", cap.stdout)]
ok(len(fe) == 4 and all(abs(a - b) < 5 for a, b in zip(fe, [4000, 6000, 2000, 2000])), f"[B] (e) の 4 行が 4000・6000・2000・2000 Hz（{fe}）")
r_aa, r_naive = num(r"成分の割合：([-\d.]+) dB"), num(r"（フィルタなし ([-\d.]+) dB）")
ok(abs(r_aa + 38.1) < 1.5 and abs(r_naive + 9.8) < 1.5, f"[B] 440 の整数倍でない成分の割合が −9.8 dB と −38.1 dB（{r_naive}・{r_aa} dB）")

In [ ]:
# 保存した wav を聞く（どれも最大 0.3 以下で保存してある。音量を小さくしてから再生する）。chirp_a の後半（8〜16 kHz）は聞こえなくても音量を上げない
for f in ['chirp_a_48k.wav', 'chirp_b_direct16k.wav', 'chirp_c_naive16k.wav', 'chirp_d_aa16k.wav', 'chirp_e_direct12k.wav']:
    print(f)
    listen_file("out/" + f)

In [ ]:
# 保存した wav を聞く（どれも最大 0.3 以下で保存してある。音量を小さくしてから再生する）
for f in ['saw_16k.wav', 'saw_naive4k.wav', 'saw_aa4k.wav']:
    print(f)
    listen_file("out/" + f)

## 4. リサンプリング（IV-1-3-3、手順 C、`resample_check.py`）

自前の `resample_poly`（中身の見える実装）を scipy と照合し、44.1 kHz → 16 kHz（160/441）と 48 kHz → 16 kHz（1/3）を確かめます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 C：リサンプリング（IV-1-3-3）

  1. common.filters.resample_poly（中身の見える実装）と scipy.signal.resample_poly を照合する
  2. 44.1 kHz → 16 kHz（up/down = 160/441）と 48 kHz → 16 kHz（1/3）
  3. 既定のフィルタ（長さ 2·10·max(up,down)+1、カイザー窓 β=5）の性能：
     通過域・遷移域・阻止域の利得と、9 kHz・8.5 kHz の音がどれだけ残るか。フィルタを 10 倍長くすると
  4. librosa.resample（既定は soxr_hq）との違い
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
from common import filters as F

rng = np.random.default_rng(1)


def db(x):
    return 20 * np.log10(np.abs(x) + 1e-300)


print("1. 自前の resample_poly と scipy.signal.resample_poly")
print("   up  down   入力の長さ  出力の長さ   最大の差")
for up, down, n in ((1, 3, 48000), (3, 1, 16000), (2, 3, 9999), (160, 441, 44100), (441, 160, 16000)):
    x = rng.standard_normal(n)
    y1 = F.resample_poly(x, up, down)
    y2 = ss.resample_poly(x, up, down)
    print(f"   {up:3d} {down:5d} {n:12d} {len(y1):10d}   {np.max(np.abs(y1 - y2)):.1e}")

print("\n2. フィルタの中身（48 kHz → 16 kHz、r = max(up,down) = 3）")
r = 3
h = ss.firwin(2 * 10 * r + 1, 1.0 / r, window=("kaiser", 5.0))      # scipy の中で作られるのと同じもの
h_long = ss.firwin(2 * 100 * r + 1, 1.0 / r, window=("kaiser", 5.0))  # 10 倍長い（MATLAB の resample(…,100) と同じ長さ）
print(f"   既定のフィルタの長さ {len(h)} 点（48 kHz で {len(h) / 48:.2f} ms）、10 倍版 {len(h_long)} 点")
f_list = (4000, 6000, 7000, 7500, 8000, 8500, 9000, 10000, 12000)
for name, hh in (("既定", h), ("10倍", h_long)):
    f, H = ss.freqz(hh, 1, worN=np.array(f_list, dtype=float), fs=48000)
    print(f"   {name}: " + "  ".join(f"{int(fi)}Hz {g:6.1f}dB" for fi, g in zip(f, db(H))))
    f, H = ss.freqz(hh, 1, worN=8192, fs=48000)
    print(f"         9 kHz 以上の最大の利得 {db(H[f >= 9000]).max():6.1f} dB、"
          f"0〜6 kHz の利得の範囲 {db(H[f <= 6000]).min():+.3f}〜{db(H[f <= 6000]).max():+.3f} dB")

print("\n3. 折り返す音がどれだけ残るか（48 kHz で作った振幅 0.3 の正弦波を 16 kHz に）")
fs = 48000
t = np.arange(fs) / fs
for f0 in (9000, 8500, 12000):
    x = 0.3 * np.sin(2 * np.pi * f0 * t)
    fa = 16000 - f0 if f0 > 8000 else f0
    for name, win in (("既定", ("kaiser", 5.0)), ("10倍", h_long * 1.0)):
        y = ss.resample_poly(x, 1, 3, window=win)
        seg = y[2000:-2000] * np.hanning(len(y) - 4000)
        A = 2 * np.abs(np.sum(seg * np.exp(-2j * np.pi * fa * np.arange(len(seg)) / 16000))) / np.sum(np.hanning(len(seg)))
        print(f"   {f0} Hz → {fa} Hz に折り返す成分（{name}）：振幅 {A:.2e}（入力の {db(A / 0.3):6.1f} dB）")

print("\n4. 44.1 kHz → 16 kHz：scipy（既定）と librosa.resample（既定 soxr_hq）の違い")
import librosa
hb = ss.firwin(2001, 6000, fs=44100)
x = ss.lfilter(hb, 1, rng.standard_normal(44100 * 2))          # 6 kHz より下だけの雑音（どちらの方法も素直に通す帯域）
x = 0.3 * x / np.max(np.abs(x))
y1 = ss.resample_poly(x, 160, 441)
y1L = ss.resample_poly(x, 160, 441, window=ss.firwin(2 * 100 * 441 + 1, 1 / 441, window=("kaiser", 5.0)))
y2 = librosa.resample(x, orig_sr=44100, target_sr=16000)
n = min(len(y1), len(y2))
print(f"   長さ scipy {len(y1)}、librosa {len(y2)}")
mid = slice(3000, n - 3000)


def diff_db(u, v):
    return 10 * np.log10(np.sum((u[mid] - v[mid]) ** 2) / np.sum(u[mid] ** 2))


print(f"   差のパワー / 信号のパワー（端を除く）：scipy 既定 と librosa {diff_db(y1, y2):.1f} dB、"
      f"scipy 10倍長 と librosa {diff_db(y1L, y2):.1f} dB")
print("   → 差の主な原因は scipy の既定のフィルタの通過域の小さな波打ち（10 倍長くすると差が縮む）")
lag = np.argmax(ss.correlate(y1[mid], y2[mid], mode="full")) - (len(y1[mid]) - 1)
print(f"   相互相関の最大のずれ {lag} 標本（0 なら時刻がそろっている）")
imp = np.zeros(44100)
imp[22050] = 1.0
p1 = np.argmax(np.abs(ss.resample_poly(imp, 160, 441)))
print(f"   インパルス（44.1 kHz の 22050 番 = 0.5 秒）の行き先：scipy {p1} 番（16 kHz の 0.5 秒 = 8000 番）")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
d = [float(v) for v in re.findall(r"^\s+\d+\s+\d+\s+\d+\s+\d+\s+([\d.e+-]+)$", cap.stdout, flags=re.M)]
ok(len(d) == 5 and max(d) < 1e-13, f"[C] 1. の 5 行の差がすべて 1e-15 の桁（最大 {max(d):.1e}）")
g8, g9 = num(r"既定:.*?8000Hz\s+([-\d.]+)dB"), num(r"既定:.*?9000Hz\s+([-\d.]+)dB")
g9x = num(r"10倍:.*?9000Hz\s+([-\d.]+)dB")
ok(abs(g8 + 6.0) < 0.2 and abs(g9 + 30.6) < 0.5 and abs(g9x + 80.0) < 1.5, f"[C] 2. 既定のフィルタが 8 kHz で −6.0 dB、9 kHz で −30.6 dB、10 倍版は 9 kHz で −80.0 dB（{g8}・{g9}・{g9x}）")
l1, l10 = num(r"scipy 既定 と librosa ([-\d.]+) dB"), num(r"scipy 10倍長 と librosa ([-\d.]+) dB")
ok(abs(l1 + 61) < 3 and abs(l10 + 78) < 3, f"[C] 4. scipy と librosa の差が約 −61 dB（10 倍版で約 −78 dB）（{l1}・{l10} dB。librosa の版で変わりうる）")
ok(num(r"scipy (\d+) 番（") == 8000, "[C] 4. インパルスが 8000 番に出る")

## 5. 量子化・ディザと dB（IV-1-3-4・3-5、手順 D、`quantize_demo.py`・`db_demo.py`）

量子化誤差の分布と SNR（$6.02B + 1.76$ dB）、ディザの効果を確かめ、次に dB の計算（パワーは $10\log_{10}$、振幅は $20\log_{10}$）を数値で確かめます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 D：量子化・量子化雑音・ディザ（IV-1-3-4）

  1. 「忙しい」信号（正規分布の雑音）なら、量子化誤差は一様分布・分散 Δ²/12・白色・信号と無相関
  2. 最大振幅の正弦波の SNR を測り、6.02B + 1.76 dB と比べる（B = 4, 8, 12, 16）
  3. 式が使えない条件：小さい信号・周期的な信号（誤差が高調波になる）
  4. ディザ（TPDF）：誤差の高調波が消え、雑音の分散は Δ²/4 になる
  5. 聞く：だんだん小さくなる 440 Hz の音を 8 ビットにする（ディザなし・あり）
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
from scipy import stats
from common.audio_io import save
from common.filters import quantize

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(0)
fs = 16000


def snr_db(x, y):
    return 10 * np.log10(np.sum(x ** 2) / np.sum((y - x) ** 2))


print("1. 忙しい信号（標準偏差 0.2 の正規分布、B = 8 ビット、Δ = 2/2^8）")
B = 8
D = 2 / 2 ** B
x = np.clip(0.2 * rng.standard_normal(200000), -0.99, 0.99)
e = quantize(x, B) - x
print(f"   誤差の平均 {e.mean():+.2e}、分散 / (Δ²/12) = {e.var() / (D ** 2 / 12):.4f}")
print(f"   誤差 / Δ の一様分布 U(−1/2, 1/2) への KS 検定：p = {stats.kstest(e / D, 'uniform', args=(-0.5, 1)).pvalue:.2f}")
print(f"   信号との相関係数 {np.corrcoef(x, e)[0, 1]:+.4f}、誤差の自己相関（ずれ 1）{np.corrcoef(e[:-1], e[1:])[0, 1]:+.4f}")

print("\n2. 最大振幅の正弦波の SNR（997 Hz、振幅 1 − Δ、10 秒）")
n = np.arange(10 * fs)
print("    B   測定 [dB]   6.02B+1.76 [dB]")
for B in (4, 8, 10, 12, 14, 16):
    D = 2 / 2 ** B
    s = (1 - D) * np.sin(2 * np.pi * 997 * n / fs)
    print(f"   {B:2d}   {snr_db(s, quantize(s, B)):8.2f}   {6.02 * B + 1.76:8.2f}")

def amp_at(y, f0):
    """hann 窓をかけて、周波数 f0 の成分の振幅を読む（IV-2-3-3 のコヒーレント利得で割る）"""
    w = np.hanning(len(y))
    return 2 * np.abs(np.sum(w * y * np.exp(-2j * np.pi * f0 * np.arange(len(y)) / fs))) / np.sum(w)


print("\n3. 式が使えない条件（B = 8、Δ = 2/256 = 0.0078）：振幅 A を小さくしていく")
B = 8
D = 2 / 2 ** B
print("   A/Δ    周波数   式の予想 [dB]   測定 [dB]   誤差の分散/(Δ²/12)   3倍の高調波 [dBc]")
CASES = [(r, f) for r in (100, 10, 3, 1.28, 0.6, 0.4) for f in (997, 1000)] + [(10, 2000)]
for ratio, f0 in CASES:
    A = ratio * D
    s = A * np.sin(2 * np.pi * f0 * n / fs)
    y = quantize(s, B)
    pred = 6.02 * B + 1.76 + 20 * np.log10(A)        # 振幅が A 倍なら信号のパワーが A² 倍
    h3 = amp_at(y, 3 * f0) / amp_at(y, f0) if np.any(y) else 0.0
    h3s = f"{20 * np.log10(h3):7.1f}" if h3 > 0 else "  （出力が全部 0）"
    print(f"   {ratio:5.2f}  {f0:5d}   {pred:10.2f}   {snr_db(s, y):10.2f}   {(y - s).var() / (D ** 2 / 12):12.2f}"
          f"        {h3s}")
print("   （1000 Hz は 16 点で1周期、最後の行の 2000 Hz は 8 点で1周期なので、誤差も同じ周期の波形＝高調波だけになる）")

print("\n4. ディザ（A = 1.28Δ ≈ −40 dBFS、1000 Hz、B = 8）")
s = 10 ** (-40 / 20) * np.sin(2 * np.pi * 1000 * n / fs)
for dith in (None, "rect", "tpdf"):
    y = quantize(s, B, dither=dith, rng=np.random.default_rng(1))
    err = y - s
    h = [20 * np.log10(amp_at(y, k * 1000) / amp_at(y, 1000) + 1e-12) for k in (3, 5)]
    print(f"   ディザ {str(dith):5s}: 誤差の分散/(Δ²/12) = {err.var() / (D ** 2 / 12):.2f}、"
          f"出力の 3倍・5倍の高調波 {h[0]:6.1f}・{h[1]:6.1f} dBc、SNR {snr_db(s, y):5.2f} dB")
print("   （理論：ディザなしは誤差が高調波になる。rect は誤差の平均が信号によらず 0 だが分散が信号で変わる。"
      "tpdf は分散も Δ²/4（Δ²/12 の 3 倍）で一定）")
# 誤差の「平均と分散が信号によらないか」を、信号の値ごとに確かめる
xs = rng.uniform(-3 * D, 3 * D, 400000)
for dith in (None, "rect", "tpdf"):
    err = quantize(xs, B, dither=dith, rng=np.random.default_rng(2)) - xs
    bins = np.floor((xs / D + 3) * 4).astype(int)        # Δ/4 ごとの区間
    m = [err[bins == k].mean() / D for k in range(24)]
    v = [err[bins == k].var() / D ** 2 for k in range(24)]
    print(f"   信号の値ごとの誤差（{str(dith):5s}）：平均/Δ の範囲 {min(m):+.3f}〜{max(m):+.3f}、"
          f"分散/Δ² の範囲 {min(v):.3f}〜{max(v):.3f}")

print("\n5. 聞く：440 Hz の音が −11 dBFS から −71 dBFS まで 4 秒で小さくなる（8 ビット）")
t = np.arange(4 * fs) / fs
level_db = -11 - 60 * t / 4                      # 量子化やディザで最大値が少し増えても 0.3 以下に収まるように −11 から
s = 10 ** (level_db / 20) * np.sin(2 * np.pi * 440 * t)
save(OUT / "fade_float.wav", s, fs)
for dith in (None, "tpdf"):
    y = quantize(s, 8, dither=dith, rng=np.random.default_rng(3))
    save(OUT / f"fade_8bit_{'tpdf' if dith else 'nodither'}.wav", y, fs)
    zero_from = t[np.nonzero(y)[0][-1]] if np.any(y) else 0
    print(f"   ディザ {str(dith):5s}: 最後に 0 でない値が出る時刻 {zero_from:.2f} s、"
          f"最後の 1 秒の SNR {snr_db(s[-fs:], y[-fs:]):6.2f} dB")
print(f"   wav：{OUT}/fade_float.wav・fade_8bit_nodither.wav・fade_8bit_tpdf.wav")
print("\n参考：有効ビット数（ENOB）＝（測った SNR − 1.76）/ 6.02。"
      f"Pico の ADC を約 9 ビットとすると SNR は約 {6.02 * 9 + 1.76:.1f} dB")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
v = num(r"分散 / \(Δ²/12\) = ([\d.]+)")
p_ks = num(r"KS 検定：p = ([\d.]+)")
ok(abs(v - 1) < 0.03 and p_ks > 0.05, f"[D] 1. の分散の比が約 1.00（{v}）、KS 検定の p が 0.05 より大きい（{p_ks}）")
snr = [(int(b), float(m), float(t)) for b, m, t in re.findall(r"^\s+(\d+)\s+([\d.]+)\s+([\d.]+)$", cap.stdout, flags=re.M)]
ok(len(snr) == 6 and all(abs(m - t) < 1 for b, m, t in snr), f"[D] 2. の 6 行が式と 1 dB 以内（{snr}）")
last = [float(x) for x in re.search(r"^\s+10\.00\s+2000\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+([-\d.]+)$", cap.stdout, flags=re.M).groups()]
ok(abs(last[0] - 27.78) < 0.01 and abs(last[1] - 42.97) < 1 and last[2] < 0.1 and abs(last[3] + 45.9) < 2,
   f"[D] 3. の最後の行（A/Δ = 10・2000 Hz）が「予想 27.78、測定 42.97、分散の比 0.03、3倍の高調波 −45.9 dBc」前後（{last}）")
vt = num(r"ディザ tpdf : 誤差の分散/\(Δ²/12\) = ([\d.]+)")
h3, h5 = num(r"ディザ tpdf : .*?高調波\s+([-\d.]+)・"), num(r"ディザ tpdf : .*?高調波\s+[-\d.]+・\s*([-\d.]+) dBc")
ok(abs(vt - 3) < 0.15 and h3 < -50 and h5 < -50, f"[D] 4. TPDF の分散の比が約 3.0（{vt}）、高調波が −50 dBc より下（{h3}・{h5} dBc）")
tn = num(r"ディザ None : 最後に 0 でない値が出る時刻 ([\d.]+) s")
ok(abs(tn - 2.48) < 0.05, f"[D] 5. ディザなしは約 2.48 秒で途切れる（{tn} s）")

In [ ]:
# 保存した wav を聞く（どれも最大 0.3 以下で保存してある。音量を小さくしてから再生する）。ディザの有無はイヤホンのほうが分かりやすい（小さな音量から）
for f in ['fade_float.wav', 'fade_8bit_nodither.wav', 'fade_8bit_tpdf.wav']:
    print(f)
    listen_file("out/" + f)

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 D：dB の計算を数値で確かめる（IV-1-3-5）

  1. パワーの比は 10log10、振幅の比は 20log10
  2. 無相関な音の足し算（パワーが足される）と、同じ位相の音の足し算（振幅が足される）
  3. dBFS：最大値で測るか、実効値で測るか（正弦波では 3.01 dB 違う）
  4. dB SPL：基準 20 µPa。1 Pa は何 dB SPL か。マイクの感度（例の値）から dBFS と dB SPL をつなぐ
"""
import numpy as np

rng = np.random.default_rng(0)
print("1. 比と dB")
for r in (2, 10, 0.5, 1000):
    print(f"   振幅 {r} 倍 = パワー {r ** 2:g} 倍 = {20 * np.log10(r):+.2f} dB（10log10({r ** 2:g}) = {10 * np.log10(r ** 2):+.2f}）")

print("\n2. 音の足し算（同じ大きさの2つの音）")
n = 160000
a = rng.standard_normal(n)
b = rng.standard_normal(n)                     # a と無相関
p = lambda x: 10 * np.log10(np.mean(x ** 2))
print(f"   無相関な雑音：{p(a):.2f} dB と {p(b):.2f} dB を足すと {p(a + b):.2f} dB（+{p(a + b) - p(a):.2f} dB、理論 +3.01）")
s = np.sin(2 * np.pi * 1000 * np.arange(n) / 16000)
print(f"   同じ位相の正弦波：{p(s):.2f} dB を2つ足すと {p(2 * s):.2f} dB（+{p(2 * s) - p(s):.2f} dB、理論 +6.02）")
s2 = np.sin(2 * np.pi * 1000 * np.arange(n) / 16000 + np.pi)
print(f"   逆の位相の正弦波：足すと {np.max(np.abs(s + s2)):.1e}（打ち消し合って 0）")
print(f"   60 dB と 50 dB（無相関）の和：{10 * np.log10(10 ** 6 + 10 ** 5):.2f} dB（小さいほうは 0.41 dB しか効かない）")

print("\n3. dBFS（最大値の 1.0 を 0 dBFS とする）")
x = 0.5 * np.sin(2 * np.pi * 1000 * np.arange(16000) / 16000)
peak = 20 * np.log10(np.max(np.abs(x)))
rms = 20 * np.log10(np.sqrt(np.mean(x ** 2)))
print(f"   振幅 0.5 の正弦波：最大値で {peak:.2f} dBFS、実効値で {rms:.2f} dBFS（差 {peak - rms:.2f} dB = 20log10√2）")
print("   実効値で測るとき、最大振幅の正弦波を 0 dBFS とする流儀（AES17 など。規格の文言は要確認）では +3.01 dB して "
      f"{rms + 20 * np.log10(np.sqrt(2)):.2f} dBFS。どちらの流儀かを必ず書く")

print("\n4. dB SPL（基準 p0 = 20 µPa、実効値の音圧で）")
p0 = 20e-6
for pa in (20e-6, 0.02, 1.0, 20.0):
    print(f"   {pa:g} Pa → {20 * np.log10(pa / p0):6.2f} dB SPL")
sens = -26.0                                    # 例：94 dB SPL（1 Pa）の正弦波で −26 dBFS（実効値）になるマイク。実物はデータシートで要確認
for spl in (94, 60, 40):
    print(f"   感度 {sens} dBFS/Pa の例のマイクで {spl} dB SPL → {sens + (spl - 94):.0f} dBFS（増幅の設定が変われば全部ずれる）")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
u, c = num(r"（\+([\d.]+) dB、理論 \+3\.01）"), num(r"（\+([\d.]+) dB、理論 \+6\.02）")
ok(abs(u - 3.01) < 0.1 and abs(c - 6.02) < 0.01, f"[D] 無相関な雑音の和が +3.02 dB 前後（{u}）、同じ位相が +6.02 dB（{c}）")
pk, rms = num(r"最大値で ([-\d.]+) dBFS、実効値で"), num(r"実効値で ([-\d.]+) dBFS（差")
ok(abs(pk + 6.02) < 0.01 and abs(rms + 9.03) < 0.01, f"[D] 振幅 0.5 の正弦波が最大値で −6.02 dBFS、実効値で −9.03 dBFS（{pk}・{rms}）")
ok(abs(num(r"^\s+1 Pa →\s+([\d.]+) dB SPL") - 93.98) < 0.01, "[D] 1 Pa が 93.98 dB SPL")

## 6. LTI システムとたたみ込み（IV-1-3-6、手順 E、`lti_conv.py`）

たたみ込みの 4 つの計算（定義・行列・FFT・ライブラリ）を照合し、pyroomacoustics で作った部屋のインパルス応答を声にたたみ込みます。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 E：LTI システムとたたみ込み（IV-1-3-6）

  1. たたみ込みの4つの計算（定義・行列・FFT・ライブラリ）を照合し、時間を比べる
  2. FFT の長さが足りないと円状たたみ込み（回り込み）になる
  3. 線形性・時不変性の判定（フィルタ・2乗・変調（ロックインの掛け算））
  4. 因果性と安定性：h[n] = a^n u[n] の Σ|h| と、有界な入力への出力
  5. ロックイン（第3章）は「複素の帯域通過フィルタ」の出力の大きさと同じ
  6. pyroomacoustics の部屋のシミュレーション ＝ 部屋のインパルス応答とのたたみ込み（自前と照合）
  wav：out/rir_dry.wav（元）・out/rir_wet.wav（部屋を通した音）・out/rir.wav（インパルス応答）
"""
import pathlib
import sys
import time

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
from common import filters as F
from common.audio_io import save

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(0)

print("1. たたみ込み y = h * x の4つの計算（x は 2000 点、h は 300 点）")
x = rng.standard_normal(2000)
h = rng.standard_normal(300) * np.exp(-np.arange(300) / 60)
ref = np.convolve(x, h)
for name, fn in (("定義（二重ループ）", lambda: F.conv_direct(x, h)),
                 ("行列 C @ x", lambda: F.conv_matrix(h, len(x)) @ x),
                 ("FFT（自前）", lambda: F.conv_fft(x, h)),
                 ("scipy fftconvolve", lambda: ss.fftconvolve(x, h)),
                 ("scipy oaconvolve", lambda: ss.oaconvolve(x, h))):
    t0 = time.perf_counter()
    y = fn()
    dt = time.perf_counter() - t0
    print(f"   {name:18s}: numpy.convolve との最大の差 {np.max(np.abs(y - ref)):.1e}、時間 {dt * 1e3:8.2f} ms")
C = F.conv_matrix([1, 2, 3], 4)
print("   h = [1, 2, 3]、長さ 4 の入力のたたみ込み行列（テプリッツ行列）：")
for row in C.astype(int):
    print("      " + " ".join(f"{v:2d}" for v in row))

print("\n2. FFT の長さと回り込み（x = [1, 2, 3, 4]、h = [1, 1, 1]、正しい長さは 4+3−1 = 6）")
xs, hs = np.array([1., 2, 3, 4]), np.array([1., 1, 1])
print(f"   線形たたみ込み      {np.convolve(xs, hs).astype(int).tolist()}")
for N in (4, 5, 6, 8):
    yc = np.real(np.fft.ifft(np.fft.fft(xs, N) * np.fft.fft(hs, N)))
    print(f"   FFT の長さ N = {N}：{np.round(yc).astype(int).tolist()}")
print("   → N が 6 より短いと、はみ出た後ろの値が前に足される（円状たたみ込み）")

print("\n3. 線形性・時不変性の判定（乱数の入力で数値的に）")
b, a = ss.butter(2, 1000, fs=16000)
w0 = 2 * np.pi * 1000 / 16000
systems = {
    "2次の低域通過フィルタ": lambda u: ss.lfilter(b, a, u),
    "2乗 y = x²": lambda u: u ** 2,
    "変調 y = x·cos(ω0 n)": lambda u: u * np.cos(w0 * np.arange(len(u))),
}
x1, x2 = rng.standard_normal(500), rng.standard_normal(500)
for name, S in systems.items():
    lin = np.max(np.abs(S(2 * x1 + 3 * x2) - (2 * S(x1) + 3 * S(x2))))
    d = 7
    shifted_in = np.concatenate([np.zeros(d), x1])[:500]
    shifted_out = np.concatenate([np.zeros(d), S(x1)])[:500]
    ti = np.max(np.abs(S(shifted_in) - shifted_out))
    print(f"   {name:16s}: 線形性のずれ {lin:.1e}、時不変性のずれ {ti:.1e} → "
          f"{'LTI' if lin < 1e-9 and ti < 1e-9 else 'LTI ではない'}")

print("\n4. 安定性：h[n] = a^n（n ≥ 0）。Σ|h| = 1/(1−|a|)（|a| < 1 のとき）")
for a_ in (0.5, 0.9, -0.9, 0.99, 1.0, 1.01):
    hh = a_ ** np.arange(3000)
    y = ss.lfilter([1.0], [1.0, -a_], np.sign(rng.standard_normal(3000)))   # ±1 の有界な入力
    th = f"{1 / (1 - abs(a_)):.1f}" if abs(a_) < 1 else "∞"
    print(f"   a = {a_:+.2f}：Σ|h|（3000 項）= {np.sum(np.abs(hh)):10.3g}（理論 {th:>6s}）、"
          f"出力の最大の大きさ {np.max(np.abs(y)):10.3g}")
y = ss.lfilter([1.0], [1.0, -1.0], np.ones(3000))
print(f"   a = 1 に「ずっと 1」を入れると出力は {y[-1]:.0f}（n に比例して増え続ける：有界でない）")
print("   移動平均を「前後に」とる h[n] = 1/5（n = −2〜2）は安定だが、n < 0 に値があるので因果的でない"
      "（未来の入力が要る。録音を後から処理するなら使える）")

print("\n5. ロックイン ＝ 複素の帯域通過フィルタ（第3章 3-3-3）")
fs = 16000
Nw = 160                                         # 10 ms（1000 Hz なら 10 周期）
n = np.arange(4000)
sig = 0.7 * np.cos(2 * np.pi * 1000 * n / fs + 0.4) + 0.3 * np.cos(2 * np.pi * 3000 * n / fs) + 0.05 * rng.standard_normal(len(n))
# (a) ロックイン：e^{−jω0 k} を掛けて、直近 Nw 点で平均（× 2）
lock = np.array([2 / Nw * np.sum(sig[m - Nw + 1:m + 1] * np.exp(-1j * w0 * np.arange(m - Nw + 1, m + 1)))
                 for m in range(Nw - 1, len(n))])
# (b) LTI のフィルタ：h[m] = (2/Nw)·e^{+jω0 m}（m = 0, …, Nw−1）を、そのままたたみ込む
hb = 2 / Nw * np.exp(1j * w0 * np.arange(Nw))
filt = np.convolve(sig, hb)[Nw - 1:len(n)]
print(f"   |ロックインの出力| と |帯域通過フィルタの出力| の最大の差 {np.max(np.abs(np.abs(lock) - np.abs(filt))):.1e}")
print(f"   ロックインの振幅の平均 {np.mean(np.abs(lock)):.4f}（入れた 1000 Hz の振幅 0.7）、"
      f"位相の平均 {np.angle(np.mean(lock)):.3f} rad（入れた位相 0.4）")
fr, H = F.freqz(hb, 1.0, n=8000, fs=fs)
for f0 in (1000, 1100, 2000, 3000, 1050):
    g = abs(H[np.argmin(np.abs(fr - f0))])
    print(f"   このフィルタの利得 {f0} Hz：" + (f"{20 * np.log10(g):6.1f} dB" if g > 1e-9 else "0（丸め誤差の大きさ）"))
print("   → 1000 Hz で 2 倍（+6.0 dB：cos の振幅の半分ずつが ±1000 Hz にあるので、2 倍して振幅に戻す）。"
      "10 ms の窓は 1000 Hz から 100 Hz おきに 0 を持つ（整数周期の平均で高調波が消える理由）")

print("\n6. 部屋のインパルス応答（pyroomacoustics、16 kHz、5×4×3 m、残響時間 0.3 s のつもり）")
import pyroomacoustics as pra
sys.path.insert(0, ".")   # ノートブック版：sounds.py はこのノートブックの隣
try:
    from sounds import speech                    # IV-2 の音（声）
    dry, name = speech(fs, sec=3.0)
except ImportError:
    print("   IV-2 の sounds.py が見つからないので、雑音のバーストを使います")
    dry = np.zeros(3 * fs)
    dry[:fs // 2] = 0.3 * rng.standard_normal(fs // 2)
    name = "noise"
dry = np.asarray(dry, dtype=float)               # librosa の読み込みは float32 なので float64 にそろえる
dim = [5.0, 4.0, 3.0]
e_abs, max_order = pra.inverse_sabine(0.3, dim)
room = pra.ShoeBox(dim, fs=fs, materials=pra.Material(e_abs), max_order=max_order)
src, mic = np.array([1.5, 1.5, 1.2]), np.array([3.5, 2.5, 1.5])
room.add_source(src.tolist(), signal=dry)
room.add_microphone(mic.tolist())
room.compute_rir()
room.simulate()
rir = room.rir[0][0]
wet_pra = room.mic_array.signals[0]
wet_own = F.conv_fft(dry, rir)
print(f"   音源：{name}、インパルス応答の長さ {len(rir)} 点（{len(rir) / fs:.3f} s）")
print(f"   room.simulate() と、自前の FFT たたみ込みの差：最大 {np.max(np.abs(wet_pra[:len(wet_own)] - wet_own)):.1e}"
      f"（長さ {len(wet_pra)} と {len(wet_own)}）")
dist = np.linalg.norm(src - mic)
c = room.c
print(f"   直接音：距離 {dist:.3f} m、音速 {c:.1f} m/s → {dist / c * 1e3:.3f} ms ＝ {dist / c * fs:.1f} 標本。"
      f"インパルス応答の最大の位置 {np.argmax(np.abs(rir))} 標本")
print(f"   差 {np.argmax(np.abs(rir)) - dist / c * fs:.1f} 標本は、分数遅延フィルタ（長さ "
      f"{pra.constants.get('frac_delay_length')}）の中心の分だけ後ろにずれているため（版で変わりうる）")
print(f"   Σ|h| = {np.sum(np.abs(rir)):.3f}（有限なので安定）、h[n] は n < 0 で 0（因果的）")
pk = max(np.max(np.abs(dry)), np.max(np.abs(wet_own)))
save(OUT / "rir_dry.wav", dry / pk * 0.3, fs)
save(OUT / "rir_wet.wav", wet_own / pk * 0.3, fs)
save(OUT / "rir.wav", rir / np.max(np.abs(rir)) * 0.3, fs)
print(f"   wav を保存しました：{OUT}/rir_dry.wav・rir_wet.wav・rir.wav")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
d = [float(v) for v in re.findall(r"numpy\.convolve との最大の差 ([\d.e+-]+)", cap.stdout)]
ok(len(d) == 5 and max(d) < 1e-12, f"[E] 1. の 5 行の差が 1e-14 以下（最大 {max(d):.1e}）")
circ = {int(n): [int(v) for v in body.split(",")] for n, body in re.findall(r"FFT の長さ N = (\d+)：\[([^\]]*)\]", cap.stdout)}
lin = [1, 3, 6, 9, 7, 4]
ok(circ[4] != lin[:4] and circ[5] != lin[:5] and circ[6] == lin and circ[8][:6] == lin, f"[E] 2. で N = 4, 5 が回り込み、N = 6, 8 が正しい（{circ}）")
jd = re.findall(r"→ (LTI(?: ではない)?)$", cap.stdout, flags=re.M)
ok(jd == ["LTI", "LTI ではない", "LTI ではない"], f"[E] 3. の判定が「LTI」「LTI ではない」「LTI ではない」（{jd}）")
dl = num(r"帯域通過フィルタの出力\| の最大の差 ([\d.e+-]+)")
g1k = num(r"このフィルタの利得 1000 Hz：\s*([-\d.]+) dB")
zeros = all(f"このフィルタの利得 {f} Hz：0（" in cap.stdout for f in (1100, 2000, 3000))
ok(dl < 1e-13 and abs(g1k - 6.0) < 0.1 and zeros, f"[E] 5. の差が 1e-15 の桁（{dl:.1e}）、利得が 1000 Hz で 6.0 dB（{g1k}）、1100・2000・3000 Hz で 0")
dr, pos = num(r"自前の FFT たたみ込みの差：最大 ([\d.e+-]+)"), num(r"インパルス応答の最大の位置 (\d+) 標本")
ok(dr < 1e-13 and abs(pos - 145) <= 5, f"[E] 6. の差が 1e-16 の桁（{dr:.1e}）、最大の位置が 145 標本前後（{pos:.0f}。pyroomacoustics の版で変わりうる）")

In [ ]:
# 保存した wav を聞く（どれも最大 0.3 以下で保存してある。音量を小さくしてから再生する）
for f in ['rir_dry.wav', 'rir_wet.wav', 'rir.wav']:
    print(f)
    listen_file("out/" + f)

## 7. z 変換・極と零点・周波数応答（IV-1-3-7、手順 F、`zplane_demo.py`）

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 F：z 変換・極と零点・周波数応答（IV-1-3-7）

  1. 手計算の例 H(z) = (1 + z⁻¹)/(1 − 0.9z⁻¹)：極と零点、|H| を距離の比で求める
  2. 部分分数：H(z) = 1/((1 − 0.9z⁻¹)(1 + 0.5z⁻¹)) のインパルス応答を式で出し、lfilter・residuez と照合
  3. 2次の共振器（IV-2 の合成の声のフォルマント 700 Hz・帯域幅 80 Hz）：極の半径・角度と、山の位置・幅
  4. 自前の freqz・lfilter・極と零点（np.roots）を scipy と照合。DTFT を N 点で取り出すと DFT
  5. 極が単位円の外にあると発散する
  out/zplane.png：共振器の極と零点、振幅特性
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
from common import plotting  # noqa: F401
from common import filters as F

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
fs = 16000

print("1. H(z) = (1 + z⁻¹)/(1 − 0.9z⁻¹)")
b1, a1 = np.array([1.0, 1.0]), np.array([1.0, -0.9])
z1, p1 = np.roots(b1), np.roots(a1)
print(f"   零点 {z1.real.tolist()}、極 {p1.real.tolist()}")
for w in (0.0, np.pi / 2, np.pi * 0.999):
    e = np.exp(1j * w)
    geo = abs(e - z1[0]) / abs(e - p1[0])           # |H| = （零点までの距離）/（極までの距離）
    Hd = (1 + np.exp(-1j * w)) / (1 - 0.9 * np.exp(-1j * w))
    print(f"   ω = {w / np.pi:.3f}π：距離の比 {geo:8.4f}、式に代入 {abs(Hd):8.4f}（{20 * np.log10(abs(Hd)):7.2f} dB）")

print("\n2. 部分分数 H(z) = 1/((1 − 0.9z⁻¹)(1 + 0.5z⁻¹)) = A/(1 − 0.9z⁻¹) + B/(1 + 0.5z⁻¹)")
a2 = np.convolve([1, -0.9], [1, 0.5])
A, B = 0.9 / 1.4, 0.5 / 1.4
print(f"   a = {a2.tolist()}、A = 0.9/1.4 = {A:.6f}、B = 0.5/1.4 = {B:.6f}")
n = np.arange(30)
h_formula = A * 0.9 ** n + B * (-0.5) ** n
h_lfilter = F.lfilter([1.0], a2, (n == 0).astype(float))
r, p, k = ss.residuez([1.0], a2)
print(f"   h[0..4] 式 {np.round(h_formula[:5], 6).tolist()}")
print(f"   式と自前 lfilter の差 {np.max(np.abs(h_formula - h_lfilter)):.1e}")
print(f"   scipy.signal.residuez：留数 {np.round(np.real(r), 6).tolist()}、極 {np.round(np.real(p), 6).tolist()}")

print("\n3. 2次の共振器（IV-2 sounds.py の synth_vowel の 1 つめ：fc = 700 Hz、bw = 80 Hz）")
fc, bw = 700.0, 80.0
rr = np.exp(-np.pi * bw / fs)
th = 2 * np.pi * fc / fs
a3 = np.array([1.0, -2 * rr * np.cos(th), rr * rr])
b3 = np.array([1.0 - rr])
poles = np.roots(a3)
print(f"   r = exp(−π·bw/fs) = {rr:.6f}、θ = 2π·fc/fs = {th:.6f} rad")
print(f"   a = [1, −2r cosθ, r²] = {np.round(a3, 6).tolist()}")
print(f"   極 {np.round(poles, 6).tolist()}、|p| = {np.abs(poles[0]):.6f}、∠p = ±{abs(np.angle(poles[0])):.6f} rad")
f, H = F.freqz(b3, a3, n=80000, fs=fs)
mag = np.abs(H)
kpk = np.argmax(mag)
above = f[mag >= mag[kpk] / np.sqrt(2)]
print(f"   山の位置 {f[kpk]:.1f} Hz（極の角度の周波数 {fc:.1f} Hz）、"
      f"−3 dB の幅 {above.max() - above.min():.1f} Hz（作るときの bw {bw:.0f} Hz）、山の高さ {20 * np.log10(mag[kpk]):.2f} dB")
print("   近似：|p| が 1 に近いとき、山の −3 dB 幅 ≈ −ln(r)·fs/π [Hz] → "
      f"{-np.log(rr) * fs / np.pi:.1f} Hz")

print("\n4. 自前の実装と scipy の照合")
rng = np.random.default_rng(0)
b4, a4 = ss.butter(6, 2000, fs=fs)
x = rng.standard_normal(20000)
print(f"   lfilter（6 次）       ：最大の差 {np.max(np.abs(F.lfilter(b4, a4, x) - ss.lfilter(b4, a4, x))):.1e}")
print(f"   freqz（512 点）       ：最大の差 {np.max(np.abs(F.freqz(b4, a4, 512)[1] - ss.freqz(b4, a4, 512)[1])):.1e}")
z_s, p_s, k_s = ss.tf2zpk(b4, a4)
print(f"   極（np.roots と tf2zpk）：最大の差 {np.max(np.abs(np.sort_complex(np.roots(a4)) - np.sort_complex(p_s))):.1e}")
print(f"   零点は z = −1 に 6 個（丸め誤差で少し散る：−1 からの距離の最大 {np.max(np.abs(np.roots(b4) + 1)):.1e}）")
N = 64
hN = rng.standard_normal(40)                         # 長さ 40 の有限な列
dft = np.fft.fft(hN, N)
_, Hd = F.freqz(hN, 1.0, n=N // 2)                   # ω = 2πk/N（k = 0, …, N/2−1）の DTFT
print(f"   DTFT を ω = 2πk/{N} で取り出したもの と {N} 点の DFT（ゼロ詰め）の差 {np.max(np.abs(Hd - dft[:N // 2])):.1e}")

print("\n5. 極が単位円の外：a = [1, −1.05]（極 1.05）")
y = F.lfilter([1.0], [1.0, -1.05], (np.arange(200) == 0).astype(float))
print(f"   インパルス応答 h[0], h[50], h[100], h[199] = {y[0]:.2f}, {y[50]:.2f}, {y[100]:.1f}, {y[199]:.3g}（発散）")
for name, (bb, aa) in (("butter 6次 2kHz", (b4, a4)), ("共振器", (b3, a3)), ("極 1.05", ([1.0], [1.0, -1.05]))):
    print(f"   {name:14s}：極の大きさの最大 {np.max(np.abs(np.roots(aa))):.6f} → {'安定' if np.max(np.abs(np.roots(aa))) < 1 else '不安定'}")

fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
t = np.linspace(0, 2 * np.pi, 400)
ax[0].plot(np.cos(t), np.sin(t), color="0.6")
ax[0].plot(poles.real, poles.imag, "x", ms=10, mew=2, color="C3", label="極")
ax[0].plot([0, 0], [0, 0], "o", mfc="none", ms=10, color="C0", label="零点（z = 0 に 2 個）")
ax[0].set_aspect("equal")
ax[0].legend(loc="lower left")
ax[0].set_title("共振器の極と零点（単位円の内側なら安定）")
ax[0].set_xlabel("実部")
ax[0].set_ylabel("虚部")
ax[1].plot(f, 20 * np.log10(mag))
ax[1].axvline(fc, color="C3", ls="--", lw=1)
ax[1].set_xlabel("周波数 [Hz]")
ax[1].set_ylabel("利得 [dB]")
ax[1].set_title("振幅特性（極に近い周波数で山になる）")
fig.tight_layout()
fig.savefig(OUT / "zplane.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print(f"\n図を保存しました：{OUT / 'zplane.png'}")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
pairs = [(float(a), float(b)) for a, b in re.findall(r"距離の比\s+([\d.]+)、式に代入\s+([\d.]+)", cap.stdout)]
ok(len(pairs) == 3 and all(abs(a - b) < 1e-4 for a, b in pairs) and all(abs(a - t) < 1e-3 for (a, b), t in zip(pairs, [20, 1.0512, 0.0017])),
   f"[F] 1. の |H| が 20・1.0512・0（距離の比と式が同じ）（{pairs}）")
A, Bc = num(r"A = 0\.9/1\.4 = ([\d.]+)"), num(r"B = 0\.5/1\.4 = ([\d.]+)")
res = sorted(float(v) for v in re.search(r"留数 \[([^\]]+)\]", cap.stdout).group(1).split(","))
ok(abs(A - 0.642857) < 1e-6 and abs(Bc - 0.357143) < 1e-6 and all(abs(a - b) < 1e-6 for a, b in zip(res, sorted([A, Bc]))),
   f"[F] 2. の A = 0.642857、B = 0.357143 と residuez の留数が同じ（{res}）")
pk, bw = num(r"山の位置 ([\d.]+) Hz"), num(r"−3 dB の幅 ([\d.]+) Hz")
ok(abs(pk - 698.9) < 0.5 and abs(bw - 80.1) < 0.5, f"[F] 3. の山が 698.9 Hz、幅 80.1 Hz（{pk}・{bw}）")

## 8. FIR と IIR の設計（IV-1-3-8・3-9・3-11、手順 G・H、`fir_design.py`・`iir_design.py`）

窓関数法の FIR を 5 種類の窓で scipy と照合し、次にバターワースの IIR（原型の極 → 遮断周波数 → 双一次変換）を照合します。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 G：FIR フィルタの設計（窓関数法）（IV-1-3-8）

  1. 自前の firwin_lowpass と scipy.signal.firwin を、5 種類の窓で照合する
  2. 窓による違い：ギブス現象（通過域の行き過ぎ）・阻止域の最大のサイドローブ・遷移の幅。
     方形窓では N を増やしても行き過ぎ（約 9%）が減らないこと
  3. カイザー窓の設計式（次数と β）を scipy.signal.kaiserord と照合し、仕様を満たすか測る
     例：48 kHz → 16 kHz のアンチエイリアス（通過 7 kHz まで、阻止 8 kHz から、80 dB）
  4. 直線位相：係数が左右対称なら、群遅延は (N−1)/2 標本で一定
  5. 遅れの補正：遅れを補正しないと「フィルタの出力が元と違う」と誤って評価する
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
from common import filters as F

fs = 16000
fc = 2000.0

print("1. 自前の firwin_lowpass と scipy.signal.firwin（N = 101、fc = 2 kHz、fs = 16 kHz）")
for win in ("boxcar", "hann", "hamming", "blackman", ("kaiser", 8.0)):
    h1 = F.firwin_lowpass(101, fc, fs, window=win)
    h2 = ss.firwin(101, fc, window=win, fs=fs)
    print(f"   {str(win):16s}: 最大の差 {np.max(np.abs(h1 - h2)):.1e}、係数の和 {h1.sum():.12f}")


def measure(h, fc, fs):
    f, H = F.freqz(h, 1.0, n=1 << 15, fs=fs)
    g = np.abs(H)
    over = g[f <= fc].max() - 1                                  # 通過域の行き過ぎ
    i_c = np.searchsorted(f, fc)
    d = np.diff(g[i_c:])
    i_null = i_c + int(np.argmax(d > 0))                          # fc より上で最初の谷（主ローブの終わり）
    side = 20 * np.log10(g[i_null:].max())                        # 阻止域の最大のサイドローブ
    f_1db = f[np.argmax(g < 10 ** (-1 / 20))]                     # −1 dB になる周波数
    return over, side, f[i_null] - f_1db


print("\n2. 窓による違い（fc = 2 kHz、fs = 16 kHz）")
print("   窓          N    通過域の行き過ぎ   最大のサイドローブ   遷移の幅（−1 dB → 最初の谷）")
for win in ("boxcar", "hann", "hamming", "blackman", ("kaiser", 8.0)):
    for N in (51, 101, 401):
        if win != "boxcar" and N == 401:
            continue
        over, side, width = measure(ss.firwin(N, fc, window=win, fs=fs), fc, fs)
        print(f"   {str(win):14s}{N:4d}   {over * 100:8.3f} %       {side:7.1f} dB          {width:7.1f} Hz")
print("   → 方形窓（窓なし）は N を 8 倍にしても行き過ぎが約 9% のまま（ギブス現象）。幅だけが 1/N で狭まる")

print("\n3. カイザー窓の設計式：48 kHz、通過 7 kHz まで、阻止 8 kHz から、阻止域の減衰 A = 80 dB")
fs2, fp, fst, A = 48000.0, 7000.0, 8000.0, 80.0
dw = 2 * np.pi * (fst - fp) / fs2                                # 遷移の幅 [rad/標本]
N_formula = (A - 7.95) / (2.285 * dw) + 1                         # Kaiser の式（次数 + 1 = 長さ）
beta = 0.1102 * (A - 8.7) if A > 50 else (0.5842 * (A - 21) ** 0.4 + 0.07886 * (A - 21) if A >= 21 else 0.0)
numtaps_s, beta_s = ss.kaiserord(A, (fst - fp) / (fs2 / 2))
print(f"   Δω = 2π·1000/48000 = {dw:.5f} rad、長さ N = (A − 7.95)/(2.285Δω) + 1 = {N_formula:.2f} → {int(np.ceil(N_formula))}")
print(f"   β = 0.1102(A − 8.7) = {beta:.4f}")
print(f"   scipy.signal.kaiserord：長さ {numtaps_s}、β = {beta_s:.4f}")
numtaps = numtaps_s + (1 - numtaps_s % 2)                         # 奇数にそろえる（タイプ I）
h = ss.firwin(numtaps, (fp + fst) / 2, window=("kaiser", beta_s), fs=fs2)
f, H = F.freqz(h, 1.0, n=1 << 16, fs=fs2)
g = 20 * np.log10(np.abs(H) + 1e-300)
print(f"   N = {numtaps}（奇数）、遮断 7.5 kHz で設計 → 0〜7 kHz の利得 {g[f <= fp].min():+.4f}〜{g[f <= fp].max():+.4f} dB、"
      f"8 kHz 以上の最大 {g[f >= fst].max():.1f} dB")
print(f"   掛け算の数：48 kHz の 1 標本ごとに {numtaps} 回（出力を 3 個に1個しか使わないなら 1/3 で済む＝ポリフェーズ）")

print("\n4. 直線位相（N = 101、hamming）")
h = ss.firwin(101, fc, fs=fs)
print(f"   左右対称か：max|h[n] − h[N−1−n]| = {np.max(np.abs(h - h[::-1])):.1e}")
_, gd1 = F.group_delay(h, 1.0, n=512, fs=fs)
_, gd2 = ss.group_delay((h, 1.0), w=512, fs=fs)
ok = np.isfinite(gd1)
print(f"   自前の群遅延：{np.nanmin(gd1):.6f}〜{np.nanmax(gd1):.6f} 標本（(N−1)/2 = 50）、"
      f"scipy との差 {np.max(np.abs(gd1[ok] - gd2[ok])):.1e}（阻止域の零点の近くでは値が乱れる）")
f, H = F.freqz(h, 1.0, n=512, fs=fs)
A_real = H * np.exp(1j * 2 * np.pi * f / fs * 50)                 # e^{−jω·50} を取り除くと実数になるはず
print(f"   H(e^jω)·e^(+jω·50) の虚部の最大 {np.max(np.abs(A_real.imag)):.1e}（実数の振幅 × 遅れ、の形）")

print("\n5. 遅れの補正（2 kHz より下だけの雑音に、同じ低域通過フィルタをかける）")
rng = np.random.default_rng(0)
x = ss.lfilter(ss.firwin(801, 1500, fs=fs), 1.0, rng.standard_normal(fs * 2))[1000:]
y = ss.lfilter(h, 1.0, x)
snr = lambda ref, est: 10 * np.log10(np.sum(ref ** 2) / np.sum((ref - est) ** 2))
print(f"   補正なし（y と x をそのまま比べる）：SNR {snr(x[200:-200], y[200:-200]):6.2f} dB")
print(f"   50 標本ずらして比べる            ：SNR {snr(x[200:-250], y[250:-200]):6.2f} dB")
yc = np.convolve(x, h, mode="same")                               # 中心をそろえる計算（非因果的）
print(f"   numpy.convolve(mode='same')   ：SNR {snr(x[200:-200], yc[200:-200]):6.2f} dB")
print("   → 通過域の信号はほぼそのまま出てくるのに、50 標本の遅れを忘れると「ひどく歪んだ」と誤って評価する")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
d = [float(v) for v in re.findall(r"最大の差 ([\d.e+-]+)、係数の和", cap.stdout)]
ok(len(d) == 5 and max(d) < 1e-14, f"[G] 1. の差がすべて 0（最大 {max(d):.1e}）")
box = [float(v) for v in re.findall(r"^\s+boxcar\s+\d+\s+([\d.]+) %", cap.stdout, flags=re.M)]
ham = [float(v) for v in re.findall(r"^\s+hamming\s+\d+\s+[\d.]+ %\s+([-\d.]+) dB", cap.stdout, flags=re.M)]
ok(len(box) == 3 and all(7 < v < 11 for v in box), f"[G] 2. 方形窓の行き過ぎが N によらず約 8〜10%（{box}）")
ok(len(ham) == 2 and all(abs(v + 55) < 1 for v in ham), f"[G] 2. hamming のサイドローブが約 −55 dB（{ham}）")
n_eq, n_ko = num(r"長さ N = .*?→ (\d+)$"), num(r"kaiserord：長さ (\d+)")
b_eq, b_ko = num(r"β = 0\.1102\(A − 8\.7\) = ([\d.]+)"), num(r"kaiserord：長さ \d+、β = ([\d.]+)")
st = num(r"8 kHz 以上の最大 ([-\d.]+) dB")
ok(n_eq == n_ko == 242 and abs(b_eq - 7.8573) < 1e-4 and abs(b_ko - 7.8573) < 1e-4 and abs(st + 80) < 1.5,
   f"[G] 3. 長さ 242・β = 7.8573 が式と kaiserord で同じ、8 kHz 以上の最大が約 −80 dB（{st} dB）")
g1, g2 = num(r"自前の群遅延：([\d.]+)〜"), num(r"自前の群遅延：[\d.]+〜([\d.]+) 標本")
ok(abs(g1 - 50) < 1e-4 and abs(g2 - 50) < 1e-4, f"[G] 4. の群遅延が 50.000000 前後（{g1}〜{g2}）")
s0, s1 = num(r"補正なし.*?SNR\s+([-\d.]+) dB"), num(r"50 標本ずらして比べる\s*：SNR\s+([-\d.]+) dB")
ok(abs(s0 + 3.27) < 0.5 and abs(s1 - 62) < 1.5, f"[G] 5. で補正なし −3.27 dB、補正あり 62 dB 前後（{s0}・{s1}）")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 H：IIR フィルタの設計（アナログの原型 → 双一次変換）（IV-1-3-9・3-11）

  1. 手計算：1 次のバターワース低域通過（fc = 1 kHz、fs = 16 kHz）の係数を式で出し、scipy と照合
  2. 自前の butter（原型の極 → 遮断周波数 → 双一次変換）を scipy.signal.butter と照合（1〜8 次）
  3. プリワーピング：しないと −3 dB の点が下にずれる。ずれの式と測定
  4. 双一次変換は左半面を単位円の内側に写す（安定なアナログ → 安定なディジタル）
  5. 係数の表し方：高い次数・低い遮断周波数では (b, a) が数値的に不安定になる。sos（2 次の積）なら大丈夫
  6. 同じ仕様（48 kHz、通過 7 kHz、阻止 8 kHz・80 dB）に必要な次数：バターワース・チェビシェフ・楕円 と FIR
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
from common import filters as F

fs = 16000

print("1. 1 次のバターワース低域通過、fc = 1 kHz、fs = 16 kHz（プリワーピングあり）")
K = np.tan(np.pi * 1000 / fs)
b_hand = np.array([K / (1 + K), K / (1 + K)])
a_hand = np.array([1.0, (K - 1) / (K + 1)])
b_s, a_s = ss.butter(1, 1000, fs=fs)
print(f"   K = tan(π·fc/fs) = tan(π/16) = {K:.6f}")
print(f"   b = [K/(1+K), K/(1+K)] = {np.round(b_hand, 6).tolist()}、a = [1, (K−1)/(K+1)] = {np.round(a_hand, 6).tolist()}")
print(f"   scipy.signal.butter(1, 1000, fs=16000)：b = {np.round(b_s, 6).tolist()}、a = {np.round(a_s, 6).tolist()}")
f, H = F.freqz(b_hand, a_hand, n=8000, fs=fs)
print(f"   1 kHz での利得 {20 * np.log10(abs(H[np.argmin(np.abs(f - 1000))])):.3f} dB（−3.01 dB のはず）、"
      f"極 {(1 - K) / (1 + K):.6f}、零点 −1")

print("\n2. 自前の butter と scipy.signal.butter（fc = 1 kHz、fs = 16 kHz）")
for N in range(1, 9):
    for bt in ("lowpass", "highpass"):
        b1, a1 = F.butter(N, 1000, fs, btype=bt)
        b2, a2 = ss.butter(N, 1000, btype=bt, fs=fs)
        z1, p1, k1 = F.butter_zpk(N, 1000, fs, btype=bt)
        z2, p2, k2 = ss.butter(N, 1000, btype=bt, fs=fs, output="zpk")
        dp = max(np.min(np.abs(p2 - q)) for q in p1)          # それぞれの極に一番近い相手との距離
        if bt == "lowpass" or N in (1, 4, 8):
            print(f"   {N} 次 {bt:8s}: b の相対差 {np.max(np.abs(b1 - b2)) / np.max(np.abs(b2)):.1e}、"
                  f"a の差 {np.max(np.abs(a1 - a2)):.1e}、極の差 {dp:.1e}、利得 k の相対差 {abs(k1 - k2) / abs(k2):.1e}")

print("\n3. プリワーピングの有無（2 次、fs = 16 kHz）：実際に −3 dB になる周波数")
print("   設計の fc [Hz]   あり [Hz]   なし [Hz]   なしの予想 (fs/π)·atan(π·fc/fs) [Hz]")
for fc in (500, 2000, 4000, 6000, 7000):
    out = []
    for pw in (True, False):
        b, a = F.butter(2, fc, fs, prewarp=pw)
        f, H = F.freqz(b, a, n=1 << 16, fs=fs)
        g = 20 * np.log10(np.abs(H) + 1e-300)
        out.append(f[np.argmax(g < -3.0103)])
    print(f"   {fc:12d}   {out[0]:9.1f}   {out[1]:9.1f}   {fs / np.pi * np.arctan(np.pi * fc / fs):9.1f}")

print("\n4. 左半面 → 単位円の内側（乱数で 10 万点）")
rng = np.random.default_rng(0)
s = -rng.exponential(5000, 100000) + 1j * rng.normal(0, 50000, 100000)    # 実部 < 0 の点
z = (2 * fs + s) / (2 * fs - s)
print(f"   実部 < 0 の点を写した |z| の最大 {np.max(np.abs(z)):.6f}（1 より小さい）")
w = np.linspace(-0.999 * np.pi, 0.999 * np.pi, 7)
Om = 2 * fs * np.tan(w / 2)
print(f"   虚軸 s = jΩ（Ω = 2fs·tan(ω/2)）を写した |z|：{np.round(np.abs((2 * fs + 1j * Om) / (2 * fs - 1j * Om)), 12).tolist()}")
_, p8, _ = F.butter_zpk(8, 1000, fs)
print(f"   8 次・1 kHz の極の大きさの最大 {np.max(np.abs(p8)):.6f}")

print("\n5. (b, a) と sos（48 kHz、低域通過）：インパルス応答の最大値")
x = np.zeros(48000 * 2)
x[0] = 1.0
print("   次数  fc [Hz]   np.roots(a) の |p| の最大   lfilter(b, a)    sosfilt(sos)")
for N, fc in ((4, 100), (6, 50), (8, 200), (8, 100), (10, 100)):
    b, a = ss.butter(N, fc, fs=48000)
    sos = ss.butter(N, fc, fs=48000, output="sos")
    with np.errstate(all="ignore"):
        y1 = ss.lfilter(b, a, x)
    y2 = ss.sosfilt(sos, x)
    m1 = np.max(np.abs(y1)) if np.all(np.isfinite(y1)) else float("inf")
    print(f"   {N:4d} {fc:8d}   {np.max(np.abs(np.roots(a))):18.6f}   {m1:13.4g}   {np.max(np.abs(y2)):12.4g}")
print("   → 本当の極はすべて単位円の内側なのに、多項式の係数に丸めると根が外に出る（8 次・100 Hz で発散）")
_, p_true, _ = ss.butter(8, 100, fs=48000, output="zpk")
print(f"   8 次・100 Hz の本当の極の大きさの最大 {np.max(np.abs(p_true)):.6f}（zpk で直接求めたもの）")

print("\n6. 同じ仕様（48 kHz、通過 0〜7 kHz で 1 dB 以内、阻止 8 kHz 以上で 80 dB）に必要な次数")
for name, fn in (("バターワース", ss.buttord), ("チェビシェフ I 型", ss.cheb1ord), ("楕円", ss.ellipord)):
    N, _ = fn(7000, 8000, 1, 80, fs=48000)
    print(f"   {name:12s}: {N:3d} 次（2 次の部分が {int(np.ceil(N / 2))} 個、1 標本あたりの掛け算 約 {5 * int(np.ceil(N / 2))} 回）")
numtaps, beta = ss.kaiserord(80, 1000 / 24000)
print(f"   FIR（カイザー窓）   : 長さ {numtaps}（1 標本あたり約 {numtaps} 回、直線位相）")
N, Wn = ss.ellipord(7000, 8000, 1, 80, fs=48000)
sos = ss.ellip(N, 1, 80, Wn, fs=48000, output="sos")
f = np.linspace(0, 24000, 4096, endpoint=False)
gd = sum(ss.group_delay((sec[:3], sec[3:]), w=f, fs=48000)[1] for sec in sos)   # 2 次の部分の群遅延の和
print(f"   楕円フィルタの群遅延：1 kHz で {gd[np.argmin(np.abs(f - 1000))]:.1f} 標本、"
      f"6.9 kHz で {gd[np.argmin(np.abs(f - 6900))]:.1f} 標本（周波数で大きく変わる＝位相がひずむ）")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
K = num(r"K = tan\(π·fc/fs\) = tan\(π/16\) = ([\d.]+)")
sb = re.search(r"scipy\.signal\.butter\(1, 1000, fs=16000\)：b = \[([\d.]+), ([\d.]+)\]、a = \[[\d.]+, ([-\d.]+)\]", cap.stdout).groups()
ok(abs(K - 0.198912) < 1e-6 and abs(float(sb[0]) - 0.165911) < 1e-6 and abs(float(sb[2]) + 0.668179) < 1e-6,
   f"[H] K = 0.198912、b = [0.165911, 0.165911]、a = [1, −0.668179]（scipy {sb}）")
dd = [float(v) for row in re.findall(r"b の相対差 ([\d.e+-]+)、a の差 ([\d.e+-]+)、極の差 ([\d.e+-]+)、利得 k の相対差 ([\d.e+-]+)", cap.stdout) for v in row]
ok(len(dd) == 44 and max(dd) < 1e-13, f"[H] 2. の差がすべて 1e-14 以下（最大 {max(dd):.1e}）")
pw = [tuple(float(v) for v in r) for r in re.findall(r"^\s+(\d+)\s+([\d.]+)\s+([\d.]+)\s+([\d.]+)$", cap.stdout, flags=re.M)]
ok(len(pw) == 5 and all(abs(a - fc) < 0.5 and abs(n - pr) < 0.5 for fc, a, n, pr in pw), f"[H] 3. の「あり」が設計の fc、「なし」の列が式と一致（{pw}）")
r8 = re.search(r"^\s+8\s+100\s+([\d.]+)\s+(\S+)\s+([\d.]+)$", cap.stdout, flags=re.M).groups()
ok(float(r8[1]) > 1e3 and abs(float(r8[2]) - 0.004415) < 1e-5, f"[H] 5. で 8 次・100 Hz の (b, a) が発散し、sos は 0.004415（{r8}）")
ok(num(r"楕円\s*:\s+(\d+) 次") == 9 and num(r"バターワース\s*:\s+(\d+) 次") == 63, "[H] 6. で楕円 9 次、バターワース 63 次")

## 9. 位相と群遅延（IV-1-3-10、手順 I、`phase_gd.py`）

群遅延、`filtfilt` のゼロ位相、全域通過フィルタを確かめます。クリック音を全域通過フィルタに通すと、振幅特性は変わらないのに音が変わります。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 I：位相と群遅延・ゼロ位相（filtfilt）・全域通過フィルタ（IV-1-3-10）

  1. バターワース（4 次・1 kHz）の群遅延：自前と scipy.signal.group_delay を照合。遮断周波数の近くで山
  2. 通過域でも群遅延が一定でないと、波形の形が変わる（FIR の直線位相と比べる）
  3. filtfilt：自前と scipy を照合。振幅は |H|²（遮断周波数で −6 dB）、位相 0、遅れ 0、ただし非因果的
  4. 全域通過フィルタ：|H| = 1 なのに音が変わる（クリックがチャープになる）。wav で聞く
  wav：out/click_orig.wav・out/click_allpass.wav、図：out/phase_gd.png
"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
import matplotlib
# matplotlib.use("Agg")   # ノートブック版：図をセルの下に表示する
import matplotlib.pyplot as plt
from common import plotting  # noqa: F401
from common import filters as F
from common.audio_io import save

OUT = pathlib.Path("out")   # ノートブック版：ノートブックの隣の out/
OUT.mkdir(exist_ok=True)
fs = 16000
rng = np.random.default_rng(0)

print("1. バターワース 4 次・1 kHz の群遅延")
b, a = ss.butter(4, 1000, fs=fs)
f1, gd1 = F.group_delay(b, a, n=4096, fs=fs)
_, gd2 = ss.group_delay((b, a), w=4096, fs=fs)
sel = f1 < 0.9 * fs / 2                                  # z = −1 の 4 重の零点の近く（ω ≈ π）は数値的に不安定なので除く
print(f"   自前と scipy の差（0〜7.2 kHz）：最大 {np.max(np.abs(gd1[sel] - gd2[sel])):.1e} 標本")
for f0 in (100, 500, 900, 1000, 2000, 4000):
    i = np.argmin(np.abs(f1 - f0))
    print(f"   {f0:5d} Hz：{gd1[i]:6.2f} 標本（{gd1[i] / fs * 1e3:5.2f} ms）")
i = np.argmax(gd1[sel])
print(f"   最大 {gd1[i]:.2f} 標本（{f1[i]:.0f} Hz、遮断周波数のすぐ下）")

print("\n2. 通過域の中の波形の形（2 kHz より下だけの雑音。周期的でないので、ずれは1通りに決まる）")
x = ss.filtfilt(ss.firwin(801, 2000, fs=fs), 1.0, rng.standard_normal(fs))
h_fir = ss.firwin(201, 2500, fs=fs)                     # 直線位相、通過域 0〜約 2 kHz
b2, a2 = ss.butter(4, 2500, fs=fs)                     # 同じくらいの通過域の IIR


def best_align_snr(ref, y, max_lag=300):
    """y を 0〜max_lag 標本ずらして ref と一番合う位置を探し、そのときの SNR [dB] とずれを返す"""
    best = (-np.inf, 0)
    seg = slice(2000, len(ref) - 2000)
    for lag in range(max_lag):
        r = ref[seg]
        e = y[2000 + lag:len(ref) - 2000 + lag]
        snr = 10 * np.log10(np.sum(r ** 2) / np.sum((r - e) ** 2))
        best = max(best, (snr, lag))
    return best


for name, (bb, aa) in (("FIR 201 点（直線位相）", (h_fir, [1.0])), ("バターワース 4 次", (b2, a2))):
    y = ss.lfilter(bb, aa, np.concatenate([x, np.zeros(400)]))
    snr, lag = best_align_snr(x, y)
    _, gd = ss.group_delay((bb, aa), w=[200, 1000, 2000], fs=fs)
    gains = np.abs(ss.freqz(bb, aa, worN=[200, 1000, 2000], fs=fs)[1])
    print(f"   {name:16s}：群遅延 200/1000/2000 Hz = {np.round(gd, 2).tolist()} 標本、"
          f"利得 {np.round(gains, 4).tolist()}、一番合うずれ {lag} 標本で SNR {snr:5.1f} dB")
Xf = np.fft.rfft(x)
_, Hb = ss.freqz(b2, a2, worN=np.fft.rfftfreq(len(x), 1 / fs), fs=fs)
y_mag = np.fft.irfft(Xf * np.abs(Hb), len(x))          # 振幅特性だけをかける（位相は 0）
seg = slice(2000, len(x) - 2000)
print(f"   参考：バターワースの振幅特性だけをかけた（位相 0）場合の SNR "
      f"{10 * np.log10(np.sum(x[seg] ** 2) / np.sum((x[seg] - y_mag[seg]) ** 2)):5.1f} dB")
print("   → 振幅だけなら SNR は高い。バターワースは周波数ごとに遅れが違うので、どうずらしても形が合わない")

print("\n3. filtfilt（前向き＋後ろ向き）")
xx = rng.standard_normal(8000)
print(f"   自前と scipy.signal.filtfilt の差：最大 {np.max(np.abs(F.filtfilt(b, a, xx) - ss.filtfilt(b, a, xx))):.1e}")
_, H = ss.freqz(b, a, worN=[1000], fs=fs)
print(f"   1 kHz の利得：1 回 {20 * np.log10(abs(H[0])):.2f} dB、filtfilt（|H|²）{40 * np.log10(abs(H[0])):.2f} dB")
step = np.concatenate([np.zeros(200), np.ones(200)])
y_lf = ss.lfilter(b, a, step)
y_ff = ss.filtfilt(b, a, step)
print(f"   ステップ（n = 200 で 0→1）：lfilter は n = 199 で {y_lf[199]:.4f}、n = 205 で {y_lf[205]:.4f}、"
      f"最大 {y_lf.max():.4f}（行き過ぎ）")
print(f"                          filtfilt は n = 195 で {y_ff[195]:.4f}（ステップの前から動く＝非因果的）、"
      f"n = 200 で {y_ff[200]:.4f}（半分の近く。左右対称なインパルス応答の和）")
xb = ss.lfilter(ss.firwin(301, 800, fs=fs), 1.0, rng.standard_normal(20000))[500:]
for name, y in (("lfilter", ss.lfilter(b, a, xb)), ("filtfilt", ss.filtfilt(b, a, xb))):
    c = ss.correlate(y, xb, mode="full")
    lag = np.argmax(c) - (len(xb) - 1)
    print(f"   800 Hz より下の雑音で、出力と入力の相互相関が最大になるずれ：{name:8s} {lag:+d} 標本")

print("\n4. 全域通過フィルタ A(z) = (a + z⁻¹)/(1 + a z⁻¹)、a = −0.6 を 300 個つなぐ")
aa1 = -0.6
f, gdA = F.group_delay([aa1, 1.0], [1.0, aa1], n=2048, fs=fs)
w = 2 * np.pi * f / fs
formula = (1 - aa1 ** 2) / (1 + 2 * aa1 * np.cos(w) + aa1 ** 2)
print(f"   1 個の群遅延：自前と式 (1−a²)/(1+2a·cosω+a²) の差 {np.max(np.abs(gdA - formula)):.1e}、"
      f"ω = 0 で {formula[0]:.2f}、ω → π で {formula[-1]:.2f} 標本")
_, HA = F.freqz([aa1, 1.0], [1.0, aa1], n=2048)
print(f"   |A(e^jω)| の範囲 {np.abs(HA).min():.12f}〜{np.abs(HA).max():.12f}（全部 1）")
M = 300
print(f"   300 個：低い音は {M * formula[0]:.0f} 標本（{M * formula[0] / fs * 1e3:.0f} ms）、"
      f"高い音は {M * formula[-1]:.0f} 標本（{M * formula[-1] / fs * 1e3:.1f} ms）遅れる")
clicks = np.zeros(int(2.5 * fs))
clicks[(np.arange(4) * 0.5 * fs + 0.2 * fs).astype(int)] = 1.0
y = clicks.copy()
for _ in range(M):
    y = ss.lfilter([aa1, 1.0], [1.0, aa1], y)
print(f"   エネルギー：入力 {np.sum(clicks ** 2):.4f}、出力 {np.sum(y ** 2):.4f}（全域通過なので同じ）")
save(OUT / "click_orig.wav", clicks * 0.3, fs)
save(OUT / "click_allpass.wav", y / np.max(np.abs(y)) * 0.3, fs)
seg = y[int(0.2 * fs):int(0.2 * fs) + 1600]
X = ss.stft(seg, fs=fs, nperseg=128, noverlap=112)
fr, tt, Z = X
mid = [fr[np.argmax(np.abs(Z[:, j]))] for j in range(len(tt))]
print(f"   1 つめのクリックの出力：時刻 {[round(t * 1e3) for t in tt[::20]]} ms の主な周波数 "
      f"{[int(m) for m in np.array(mid)[::20]]} Hz（高い音が先、低い音が後＝下がるチャープ）")
print(f"   wav：{OUT}/click_orig.wav と click_allpass.wav（音量を小さくして聞き比べる）")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(f1, gd1, label="バターワース 4 次 1 kHz")
_, gdf = F.group_delay(h_fir, 1.0, n=4096, fs=fs)
ax[0].plot(np.linspace(0, fs / 2, 4096, endpoint=False), gdf, label="FIR 201 点（直線位相）")
ax[0].set_ylim(0, 120)
ax[0].set_xlim(0, 4000)
ax[0].set_xlabel("周波数 [Hz]")
ax[0].set_ylabel("群遅延 [標本]")
ax[0].legend()
ax[1].plot(step, color="0.6", label="入力")
ax[1].plot(y_lf, label="lfilter（因果的）")
ax[1].plot(y_ff, label="filtfilt（ゼロ位相）")
ax[1].set_xlim(170, 260)
ax[1].set_xlabel("n")
ax[1].legend()
ax[2].plot(np.arange(len(seg)) / fs * 1e3, seg)
ax[2].set_xlabel("時間 [ms]")
ax[2].set_title("全域通過 300 個を通したクリック")
fig.tight_layout()
fig.savefig(OUT / "phase_gd.png", dpi=120)
plt.show()   # ノートブック版：図を表示する
print(f"図を保存しました：{OUT / 'phase_gd.png'}")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
gm, gf = num(r"最大 ([\d.]+) 標本（\d+ Hz"), num(r"最大 [\d.]+ 標本（(\d+) Hz")
ok(abs(gm - 10.1) < 0.2 and 850 < gf < 950, f"[I] 1. の群遅延の最大が約 10.1 標本（約 900 Hz）（{gm} 標本、{gf:.0f} Hz）")
sf_, sb_ = num(r"FIR 201 点.*?一番合うずれ 100 標本で SNR\s+([\d.]+) dB"), num(r"バターワース 4 次\s*：.*?SNR\s+([\d.]+) dB")
ok(abs(sf_ - 69) < 2 and abs(sb_ - 16.5) < 1.5, f"[I] 2. で FIR は 100 標本ずらして SNR 約 69 dB（{sf_}）、バターワースは最大約 16.5 dB（{sb_}）")
ff, n195 = num(r"filtfilt（\|H\|²）([-\d.]+) dB"), num(r"filtfilt は n = 195 で ([\d.]+)")
lag = num(r"相互相関が最大になるずれ：filtfilt ([+-]?\d+) 標本")
ok(abs(ff + 6.02) < 0.01 and n195 > 0.005 and lag == 0, f"[I] 3. で filtfilt の 1 kHz が −6.02 dB、n = 195 で 0 でない値（{n195}）、相互相関のずれが 0")
ein, eout = num(r"エネルギー：入力 ([\d.]+)、"), num(r"エネルギー：入力 [\d.]+、出力 ([\d.]+)")
ok(abs(ein - 4) < 1e-3 and abs(eout - 4) < 1e-3, f"[I] 全域通過のエネルギーが入力・出力とも 4.0000（{ein}・{eout}）")

In [ ]:
# 保存した wav を聞く（どれも最大 0.3 以下で保存してある。音量を小さくしてから再生する）
for f in ['click_orig.wav', 'click_allpass.wav']:
    print(f)
    listen_file("out/" + f)

## 10. 統計の誤用と、確認問題の答え合わせ（IV-1-3-12、手順 J、`stats_misuse.py`・`quiz_check.py`）

確認問題（教科書の 7 節）は、**先に紙で解いてから**最後のセルを実行してください。

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 J：この章の題材で起こる統計の誤用（IV-1-3-12）

  1. フィルタの遮断周波数と次数を、結果（p 値）を見てから選ぶ
     ：差の無い2群の「録音」を 30 通りのフィルタで比べ、一番小さい p 値を報告すると、偽陽性はどれだけ増えるか
  2. フィルタをかけた雑音は、隣どうしが相関する
     ：平均の標準誤差を s/√n で計算すると小さすぎ、95% 区間が本当の値を含む割合が 95% を大きく下回る
  （量子化の SNR の式を、小さい信号・周期的な信号に当てはめる誤用は quantize_demo.py の 3.）
"""
import numpy as np
import scipy.signal as ss
from scipy import stats

rng = np.random.default_rng(0)
fs = 16000

print("1. 遮断周波数と次数を後から選ぶ（差の無い2群、各 10 本、0.5 秒の雑音）")
n = fs // 2
freqs = np.fft.rfftfreq(n, 1 / fs)
cutoffs = np.array([300, 500, 800, 1000, 1500, 2000, 3000, 4000, 5000, 6000])
orders = (2, 4, 8)
H2 = np.stack([np.abs(ss.freqz(*ss.butter(o, c, fs=fs), worN=freqs, fs=fs)[1]) ** 2
               for o in orders for c in cutoffs], axis=1)          # (周波数, 30 通り) の |H|²
color = ss.butter(1, 2000, fs=fs)                                  # 2群とも同じ色の雑音にする


def features(n_rec):
    """n_rec 本の録音それぞれの「低域通過フィルタの出力のエネルギーの対数」（30 通り）。
    パーセバルの等式で、時間でフィルタをかける代わりに |X|²·|H|² の和で計算する"""
    x = ss.lfilter(*color, rng.standard_normal((n_rec, n)), axis=1)
    P = np.abs(np.fft.rfft(x, axis=1)) ** 2
    return np.log(P @ H2)


trials = 2000
p_fixed, p_min, p_bonf = [], [], []
for _ in range(trials):
    A, B = features(10), features(10)
    p = stats.ttest_ind(A, B, axis=0, equal_var=False).pvalue      # 30 通りの p 値
    p_fixed.append(p[len(cutoffs) * 1 + 3])                        # 事前に決めた 1 通り（4 次・1000 Hz）
    p_min.append(p.min())
    p_bonf.append(min(1.0, p.min() * len(p)))
p_fixed, p_min, p_bonf = map(np.array, (p_fixed, p_min, p_bonf))
print(f"   {trials} 回の実験で「p < 0.05」になった割合（本当は差が無いので、すべて偽陽性）")
print(f"   事前に決めた 1 通り（4 次・1 kHz）    ：{np.mean(p_fixed < 0.05):.3f}")
print(f"   30 通りから一番小さい p 値を選ぶ      ：{np.mean(p_min < 0.05):.3f}")
print(f"   30 通りの最小にボンフェローニ補正    ：{np.mean(p_bonf < 0.05):.3f}")
cor = np.corrcoef(features(400).T)
print(f"   30 通りの特徴量どうしの相関の中央値 {np.median(cor[np.triu_indices(30, 1)]):.2f}"
      "（強く相関するので、偽陽性は 1−0.95^30 = 0.785 までは増えないが、5% よりずっと大きい）")

print("\n2. フィルタをかけた雑音の平均の標準誤差（n = 2000、本当の平均は 0）")
n = 2000
for name, (b, a) in (("フィルタなし", ([1.0], [1.0])),
                     ("移動平均 16 点", (np.ones(16) / 16, [1.0])),
                     ("バターワース 2 次・500 Hz", ss.butter(2, 500, fs=fs))):
    means, naive_se, cover = [], [], []
    for _ in range(3000):
        x = ss.lfilter(b, a, rng.standard_normal(n + 500))[500:]   # 最初の 500 点は捨てる（立ち上がり）
        m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
        means.append(m)
        naive_se.append(se)
        cover.append(abs(m) < 1.96 * se)
    true_sd = np.std(means)
    x = ss.lfilter(b, a, rng.standard_normal(200000))[500:]
    xc = x - x.mean()
    S = np.abs(np.fft.rfft(xc, 2 * len(xc))) ** 2                   # 自己相関を FFT で（ゼロ詰めで回り込みなし）
    r = np.fft.irfft(S)[:101] / np.sum(xc ** 2)                     # ずれ 0〜100 の自己相関係数
    n_eff = n / (1 + 2 * np.sum(r[1:]))                             # 有効なデータの数（ずれ 100 までで打ち切り）
    print(f"   {name:22s}: s/√n の平均 {np.mean(naive_se):.4f}、本当の標準偏差 {true_sd:.4f}"
          f"（{true_sd / np.mean(naive_se):.2f} 倍）、95% 区間が 0 を含む割合 {np.mean(cover):.3f}、"
          f"有効なデータの数 n_eff ≈ {n_eff:.0f}")
print("   → 移動平均 16 点なら、だいたい 16 点で1つ分の情報しかない（n_eff ≈ 2000/16 = 125）")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
p1, pm, pb = (num(rf"{k}\s*：([\d.]+)$") for k in (r"事前に決めた 1 通り（4 次・1 kHz）", r"30 通りから一番小さい p 値を選ぶ", r"30 通りの最小にボンフェローニ補正"))
ok(abs(p1 - 0.05) < 0.015 and abs(pm - 0.15) < 0.04 and pb < 0.02, f"[J] 1. 「事前に決めた 1 通り」約 0.05、「一番小さい p 値」約 0.15、ボンフェローニ 約 0.01（{p1}・{pm}・{pb}）")
ma = re.search(r"移動平均 16 点\s*: .*?（([\d.]+) 倍）、95% 区間が 0 を含む割合 ([\d.]+)、有効なデータの数 n_eff ≈ (\d+)", cap.stdout).groups()
ok(3.5 < float(ma[0]) < 4.5 and abs(float(ma[1]) - 0.38) < 0.05 and 90 < int(ma[2]) < 150, f"[J] 2. 移動平均 16 点の行が「約 4 倍、0 を含む割合 約 0.38、n_eff 約 120」（{ma}）")

In [ ]:
%%capture cap
# ノートブック版：出力をいったん cap に受け取り、次のセルで表示してから確かめる
"""IV-1 確認問題の答え合わせ（先に紙で解いてから実行する）"""
import pathlib
import sys

sys.path.insert(0, ".")   # ノートブック版：common/ と sounds.py はこのノートブックの隣
import numpy as np
import scipy.signal as ss
from common import filters as F


def alias(f, fs):
    r = f % fs
    return min(r, fs - r)


print("問1 エイリアシング（fs = 16 kHz）")
for f in (9000, 15000, 17500, 31000):
    print(f"   {f} Hz → {alias(f, 16000):.0f} Hz")
print(f"   48 kHz の録音を 3 個に1個（フィルタなし）：10 kHz → {alias(10000, 16000):.0f} Hz")

print("問2 量子化")
for B in (16, 24):
    print(f"   {B} ビット：最大振幅の正弦波の SNR {6.02 * B + 1.76:.2f} dB")
D = 2 / 2 ** 16
print(f"   16 ビットの Δ = {D:.3e}、量子化雑音の実効値 Δ/√12 = {D / np.sqrt(12):.3e} = {20 * np.log10(D / np.sqrt(12)):.1f} dBFS")
print(f"   16 ビットで −20 dBFS の正弦波：SNR ≈ {6.02 * 16 + 1.76 - 20:.2f} dB")
print(f"   測った SNR 56 dB の有効ビット数 {(56 - 1.76) / 6.02:.2f} ビット")

print("問3 dB")
print(f"   振幅 0.1 倍 {20 * np.log10(0.1):.0f} dB、パワー 0.1 倍 {10 * np.log10(0.1):.0f} dB")
print(f"   70 dB SPL の無相関な音 2 つ：{10 * np.log10(2 * 10 ** 7):.2f} dB SPL、0.2 Pa：{20 * np.log10(0.2 / 20e-6):.2f} dB SPL")

print("問4 H(z) = (1 − z⁻²)/(1 − 0.81 z⁻²)")
b, a = [1, 0, -1], [1, 0, -0.81]
print(f"   零点 {np.roots(b).real.tolist()}、極 {np.round(np.roots(a).real, 6).tolist()}")
for w in (0.0, np.pi / 2, np.pi):
    H = np.polyval(b[::-1], np.exp(-1j * w)) / np.polyval(a[::-1], np.exp(-1j * w))
    print(f"   ω = {w / np.pi:.1f}π：|H| = {abs(H):.4f}" + (f"（{20 * np.log10(abs(H)):.2f} dB）" if abs(H) > 1e-9 else "（0）"))

print("問5 y[n] = 0.5 y[n−1] + x[n] + x[n−1]")
h = F.lfilter([1, 1], [1, -0.5], (np.arange(8) == 0).astype(float))
print(f"   h[0..5] = {np.round(h[:6], 6).tolist()}、Σ|h|（200 項）= {np.sum(np.abs(F.lfilter([1, 1], [1, -0.5], (np.arange(200) == 0).astype(float)))):.6f}")

print("問6 1 次のバターワース、fc = 4 kHz、fs = 16 kHz")
K = np.tan(np.pi * 4000 / 16000)
print(f"   プリワーピングあり：K = {K:.6f}、b = [{K / (1 + K):.6f}, {K / (1 + K):.6f}]、a = [1, {(K - 1) / (K + 1) + 0:.6f}]")
bs, as_ = ss.butter(1, 4000, fs=16000)
print(f"   scipy：b = {np.round(bs, 6).tolist()}、a = {np.round(as_, 6).tolist()}（−5.6e−17 などは丸め誤差で 0）")
Kp = 2 * np.pi * 4000 / (2 * 16000)
print(f"   なし：K' = π/4 = {Kp:.6f}、b = {round(Kp / (1 + Kp), 6)}、a1 = {round((Kp - 1) / (Kp + 1), 6)}、"
      f"−3 dB の周波数 {16000 / np.pi * np.arctan(Kp):.1f} Hz")
bb, aa = F.butter(1, 4000, 16000, prewarp=False)
f, Hh = F.freqz(bb, aa, n=1 << 16, fs=16000)
print(f"   （確かめ）プリワーピングなしの −3 dB の点 {f[np.argmax(20 * np.log10(np.abs(Hh)) < -3.0103)]:.1f} Hz")

print("問7 カイザー窓：fs = 16 kHz、遷移の幅 500 Hz、60 dB")
dw = 2 * np.pi * 500 / 16000
N = (60 - 7.95) / (2.285 * dw) + 1
print(f"   N = {N:.2f} → {int(np.ceil(N))}、β = {0.1102 * (60 - 8.7):.4f}、kaiserord {ss.kaiserord(60, 500 / 8000)}")
Nodd = int(np.ceil(N)) + (1 - int(np.ceil(N)) % 2)
print(f"   奇数にして N = {Nodd}：群遅延 {(Nodd - 1) / 2:.0f} 標本 = {(Nodd - 1) / 2 / 16:.3f} ms")

print("問8 44.1 kHz → 16 kHz")
from math import gcd
g = gcd(16000, 44100)
up, down = 16000 // g, 44100 // g
print(f"   up/down = {up}/{down}、既定のフィルタの長さ {2 * 10 * max(up, down) + 1}、"
      f"ゼロを入れた後の fs {44100 * up / 1e6:.3f} MHz、遮断 {44100 * up / 2 / max(up, down):.0f} Hz")
print(f"   10 秒（441000 点）→ {len(ss.resample_poly(np.zeros(441000), up, down))} 点")

print("問9 たたみ込み x = [1, 2, 3]、h = [1, −1]")
x, hh = np.array([1., 2, 3]), np.array([1., -1])
print(f"   線形 {np.convolve(x, hh).astype(int).tolist()}、N = 3 の円状 "
      f"{np.round(np.real(np.fft.ifft(np.fft.fft(x, 3) * np.fft.fft(hh, 3)))).astype(int).tolist()}")

print("問10 位相・群遅延・統計")
print(f"   FIR 101 点（fs = 16 kHz）の遅れ 50 標本 = {50 / 16:.3f} ms")
b4, a4 = ss.butter(4, 1000, fs=16000)
_, H1 = ss.freqz(b4, a4, worN=[1000], fs=16000)
print(f"   4 次バターワースを filtfilt：1 kHz で {40 * np.log10(abs(H1[0])):.2f} dB")
for av in (-0.5,):
    print(f"   全域通過 a = {av}：ω = 0 で {(1 - av ** 2) / (1 + 2 * av + av ** 2):.3f}、"
          f"ω = π で {(1 - av ** 2) / (1 - 2 * av + av ** 2):.3f} 標本")
L = 8
print(f"   移動平均 {L} 点の白色雑音、n = 4000：n_eff = n/(1 + 2Σ(1 − k/{L})) = "
      f"{4000 / (1 + 2 * sum(1 - k / L for k in range(1, L))):.0f}、標準誤差は s/√n の √{L} = {np.sqrt(L):.2f} 倍")

In [ ]:
cap.show()   # 上のセルの出力（文字と図）
ok, num = _ok, _num   # 確かめの関数を戻す（プログラムが ok という名前を使うことがあるので）
import re
# 確かめ（教科書の手順の「確かめること」。乱数・音源・ライブラリの版の分の幅を持たせてある）
al = [(int(a), int(b)) for a, b in re.findall(r"^\s+(\d+) Hz → (\d+) Hz$", cap.stdout, flags=re.M)]
ok(al == [(9000, 7000), (15000, 1000), (17500, 1500), (31000, 1000)], f"問1 エイリアシングの行き先（{al}）")
ok(abs(num(r"16 ビット：最大振幅の正弦波の SNR ([\d.]+) dB") - 98.08) < 0.01 and abs(num(r"測った SNR 56 dB の有効ビット数 ([\d.]+) ビット") - 9.01) < 0.01, "問2 16 ビットの SNR 98.08 dB、56 dB の有効ビット数 9.01")
ok(abs(num(r"無相関な音 2 つ：([\d.]+) dB SPL") - 73.01) < 0.01, "問3 70 dB SPL の無相関な音 2 つで 73.01 dB SPL")
ok(abs(num(r"Σ\|h\|（200 項）= ([\d.]+)") - 4) < 1e-6, "問5 Σ|h| = 4")
ok(abs(num(r"なし：K' = π/4 = [\d.]+、.*?−3 dB の周波数 ([\d.]+) Hz") - 3390.8) < 0.5, "問6 プリワーピングなしの −3 dB の周波数 約 3390.8 Hz")
ok(num(r"N = [\d.]+ → (\d+)、β") == 118 and abs(num(r"→ \d+、β = ([\d.]+)") - 5.6533) < 1e-4, "問7 カイザー窓 N = 118、β = 5.6533")
ok(abs(num(r"遅れ 50 標本 = ([\d.]+) ms") - 3.125) < 1e-6 and num(r"n_eff = .*? = (\d+)、") == 500, "問10 FIR 101 点の遅れ 3.125 ms、移動平均 8 点の n_eff = 500")

## おわりに

- 自前の `common/filters.py` が scipy・numpy と一致すること、エイリアシング・リサンプリング・量子化・たたみ込み・フィルタの設計・群遅延を、数値と音で確かめました。
- 出た数値を、教科書の 5 節の「確かめること」と見比べてください（乱数・音源・ライブラリの版で少し変わります）。
- このノートブックに入れていないもの：内蔵マイクで録音する `record_alias.py`（手順 B の実機）。Mac のターミナルで実行してください。